# <center>主流Agent类型及接口设计</center>

在《智能体项目开发必备基础FastAPI》课程中，我们把一个文本分析脚本变成了 HTTP 服务——<b>本节课，我们打算同一件事放在 agent 身上</b>,但 agent 和文本分析脚本不一样的地方在于:<b>它对外要暴露的出口不止一种</b>——不是一句"传文本进来,返结果出去"就讲完的。比如:

- 一个对话型 agent 至少要有"接收消息 + 流式回复 + 查看历史"三个出口;
- 一个工具调用 agent 还要让前端实时看到"它现在在调什么工具、调用返了什么";
- 一个文件处理 agent 要有"上传文件 + 异步解析 + 取结果"三个出口。

<div align=center><font size=2 color=#999999>本课四件事:从 agent 类型到方法论的完整路径</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-734ef0df.jpg" width=85%></div>
<br>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
.center table,
table {
margin-left: auto !important;
margin-right: auto !important;
}
.center th,
.center td,
table th,
table td {
text-align: center !important;
vertical-align: middle !important;
}
.business-outlet-table th[colspan="3"] {
font-size: 1.08em;
font-weight: 700;
}
</style>
# <center>第 1 章: 常见 Agent 类型与能力出口</center>

本章按 9 类 agent 过一遍:看产品形态、核心能力和业务出口。业务出口只用产品语言描述,不提前进入协议层。

<div align=center><font size=2 color=#999999>本章覆盖的 9 类 agent — 每类一张卡片,先看见全景,再逐节展开</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-c8fb31bc.jpg" width=85%></div>
<br>

> <font size=2>**【名词解释】<font color=red>业务接口</font>** — 站在产品角度,agent 对外提供的一项能力。例如“聊天接口”就是“用户能发送消息并收到 agent 回复”这件事本身,**不是**“用什么协议方法实现”,也不是“用哪个框架组件写”。先把业务接口列清,再翻译成具体协议和代码实现。</font>

## 1. 对话助手 agent

最基础的一类agent--对话助手。

<div align=center><font size=2 color=#999999>对话助手 agent 的典型产品形态:三栏布局 — 会话列表 + 对话主区 + 输入框</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-5b433472.jpg" width=80%></div>
<br>

打开 ChatGPT / Claude / Gemini / DeepSeek / 豆包 网页版,看到的就是对话助手 agent 的典型产品形态——左侧一栏会话列表(每条会话有个自动生成的标题),中间是对话主区(用户气泡和 agent 气泡上下排开,agent 回复字一个一个蹦出来),底部一个输入框敲文字按回车发送。聊一段时间后,你可以**点左侧任意一条历史会话切回去**,继续往下聊;也可以**新建一个会话**,从空白开始;不想要的会话可以**删掉**。

> <font size=2>**【名词解释】<font color=red>会话(Session / Conversation)</font>** — 用户和 agent 之间一段连续对话的容器。每个会话有自己的 ID(`session_id` 或 `conversation_id`)、自己的消息列表(从空开始,一条条用户消息 + agent 回复积累起来)、自己的元数据(创建时间、标题、所属用户等)。会话是产品里的**第一类业务对象**——能创建、能切换、能删除、能改标题。从接口设计角度,所有对话能力的接口都围绕"会话 ID"展开:发消息要带 session_id、拉历史要带 session_id、删会话也要 session_id。</font>

我们把这个画面拆开看,**对话助手 agent 的核心能力就两件事**:一件是**接收用户消息 + 把上下文一起喂给 LLM + 把回复一字一字流回前端**——这是聊天本身;另一件是**会话的管理**——开新会话、切回老会话、改标题、删会话。

那它对外要暴露哪些业务出口?我们照着上面的产品操作捋一遍就出来了。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>聊天接口</strong></td><td align="left">用户发一条消息,后端结合上下文生成完整回复</td><td align="left">一发一收,适合不展示中间过程</td></tr>
<tr><td align="left"><strong>流式回复接口</strong></td><td align="left">回复生成时实时把文本片段推到前端</td><td align="left">同一聊天能力的“打字机”返回方式</td></tr>
<tr><td align="left"><strong>会话历史接口</strong></td><td align="left">切回历史会话时拉取消息列表</td><td align="left">保证刷新、切换后上下文不断</td></tr>
<tr><td align="left"><strong>会话管理接口</strong></td><td align="left">新建、删除、改标题等会话对象操作</td><td align="left">管的是 session 状态,不是单条消息</td></tr>
</tbody>
</table>
</div>

四条出口里,**聊天接口和流式回复接口本质是同一件事的两种返回方式**,所以列成两条。会话历史和会话管理是“对话之外的辅助”,但**对话助手 agent 一定要有**:聊天界面如果拉不到历史,用户刷新就会丢上下文。

不过对话助手 agent 还是最"纯粹"的一类——它只负责生成文本回复，对话的功能可以说是所有agent必备的功能了。

## 2. 工具调用 agent

把对话助手 agent 升一级,让它能"对外动手",就到了工具调用 agent。

<div align=center><font size=2 color=#999999>工具调用 agent 的产品形态 — 主对话区 + 工具调用过程实时浮现:思考态 / 工具调用提示 / 结构化结果卡片</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-4765b15d.jpg" width=80%></div>
<br>

最典型的产品形态依然是 ChatGPT / Claude 等这类主流对话产品主模式默认开"工具"后的样子(再独立一些的还有 Perplexity 的搜索 + 引用流、Manus 的工具步骤面板)。你问"帮我查一下北京明天的天气",agent **不会**直接编一个回答塞给你,而是在主对话区上方/旁边**实时弹出一条灰色小提示**:`正在搜索:北京 明天 天气`——这是它在调用一个"查天气"工具;紧接着弹出一张可折叠的小卡片,展开能看到工具调用的结构化结果(`{city: '北京', date: '2026-05-12', temp: 18, condition: '晴'}`);然后 agent 才把答案文字流回主对话区:"明天北京晴,气温 18 度,适合户外活动。"

整个过程前端要展示**三件事**:① agent 正在思考的状态("我在想…");② 它调用了什么工具、参数是什么("正在查 / 已查到");③ 工具返回了什么结构化数据(以卡片形式展示)。**这是我们看到的对话助手 agent 没有的新形态**——纯对话只展示文字流;工具调用要把"它在干什么"按事件类型分别展示出来。

我们把这个产品画面里 agent 暴露的能力捋一遍,就能看到它在对话助手的基础上多了哪些出口。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>对话能力基础包</strong></td><td align="left">复用对话助手的聊天、流式、历史、会话管理四条</td><td align="left">任何带对话能力的 agent 都先继承这组出口</td></tr>
<tr><td align="left"><strong>工具调用过程接口</strong></td><td align="left">按事件推送“思考 / 工具调用 / 工具返回 / 最终回复”</td><td align="left">前端据此渲染工具状态、提示和结果卡片</td></tr>
<tr><td align="left"><strong>工具结果接口</strong></td><td align="left">按工具调用 ID 单独拉取较大或需固定展示的结果</td><td align="left">避免把大结果塞进事件流</td></tr>
</tbody>
</table>
</div>

工具调用 agent 的本质是**把“它在干什么”这件事产品化**——上一类只产品化了“它说了什么”,这一类把过程也展开给前端看。<font color=red>“事件流”是它和对话助手最大的差别</font>。

下一类我们要看的是把动作再"扎实"一点:agent 不只是调一个查天气这种短工具,而是能跑一个真正会执行几分钟到几十分钟的长任务。

## 3. 任务执行 agent

任务执行 agent 解决的是"agent 干一件不能立刻完成的事"。典型产品例子:Manus(2025 横空出世的通用自主 agent)/ OpenAI Operator(在云端浏览器里替你跑长任务)/ 智谱 AutoGLM / ChatGPT 的 Agent 模式——共同点是用户提一个目标,agent 后台跑几分钟到几十分钟才出结果。

<div align=center><font size=2 color=#999999>任务执行 agent 的产品形态 — 任务面板(任务卡片 / 进度条 / 步骤日志)与会话主区并存</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-0660d233.jpg" width=80%></div>
<br>

我们把这个画面拆开,任务执行 agent 多了一个对话助手 / 工具调用 agent 都没有的产品形态:**任务和会话是分开的两个对象**。会话承载的是"用户和 agent 聊的内容";任务承载的是"agent 在跑的长动作"——它有自己的 ID、自己的状态(待执行 / 执行中 / 已完成 / 失败)、自己的进度(0% → 100%)、自己的结果(产物文件 / 结构化输出)。前端要能在任意时刻**反过来问后端**:这个任务现在跑到哪儿了?好了吗?最终结果是什么?

我们把这套产品形态对应到对外要暴露的出口,会发现它在对话能力之外多了"任务"这个独立的对象,需要专门一组出口来管理。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>聊天能力基础包</strong></td><td align="left">复用对话助手的四条出口</td><td align="left">用户需要先能描述任务、追问任务</td></tr>
<tr><td align="left"><strong>任务创建接口</strong></td><td align="left">发起长任务,后端立刻返回任务 ID + 初始状态</td><td align="left">不阻塞等待任务真正跑完</td></tr>
<tr><td align="left"><strong>任务状态接口</strong></td><td align="left">查询执行中 / 已完成 / 失败、进度和当前步骤</td><td align="left">可轮询,也可做成后端推送</td></tr>
<tr><td align="left"><strong>任务结果接口</strong></td><td align="left">任务完成后按任务 ID 取最终产物</td><td align="left">结果可能是 JSON、文本或可下载文件</td></tr>
<tr><td align="left"><strong>任务取消接口</strong></td><td align="left">用户中途取消,后端标记并停止相关后台进程</td><td align="left">简单场景可省,正经产品建议保留</td></tr>
<tr><td align="left"><strong>任务步骤日志接口</strong></td><td align="left">实时推送任务执行过程里的步骤日志</td><td align="left">可选;代码 / Devin 类产品很常见</td></tr>
</tbody>
</table>
</div>

任务执行 agent 比工具调用 agent 又往前走了一步:**它把"长动作"从对话里独立出来,成了一个有自己生命周期的对象**。一旦产品里出现"任务卡片 / 进度 / 状态查询"这种形态,我们就要在接口设计层为它单独开一组出口——光靠聊天接口和事件流出口是兜不住的。

> <font size=2>**【名词解释】<font color=red>任务状态生命周期</font>(task state lifecycle)** — 一个长任务从创建到结束的状态流转。典型 5 个状态:**`queued`**(已提交、排队等跑)→ **`running`**(在跑、伴随进度 0% → 100%)→ **`done`**(跑完、产物可取)/ **`failed`**(挂了、带错误信息)/ **`cancelled`**(用户中途取消)。任何任务执行 / 工作流 / 长任务接口的核心都是这套状态机——前端要能在任意时刻拿到当前是哪一态。</font>

下一类我们看的是任务执行的"近亲":工作流 agent。差别在于任务执行是一个目标分成几步,**步骤是 agent 临时决定的**;工作流则是**步骤被预先编排好**,执行的是预定义的流程。

## 4. 工作流 agent

最直观的产品例子是 Dify / Coze(字节扣子)/ n8n / FastGPT / LangFlow 这类平台上的"工作流"形态。一个客服工作流大致这样设计:**用户提问 → 意图分类 → 是订单类问题就走"查订单 → 渲染订单卡片"分支,是退款类问题就走"查退款政策 → 跳转人工"分支 → 输出回复**。这套流程在产品后台先用拖拽方式画好,运行时**严格按这张图走**——每一步是哪个节点、走哪个分支,都是图本身决定的。

<div align=center><font size=2 color=#999999>工作流 agent 的产品形态(示意) — 节点画板 + 当前执行节点高亮 + 变量监视面板</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-67aec4ce.jpg" width=80%></div>
<br>

和任务执行 agent 不一样的地方是:**任务执行的"步骤"是 agent 临时决定的**(它根据当前情况自己选下一步);**工作流的"步骤"是预先编排好的**(节点和分支事先就连好了线)。这个差别在接口设计上的具体表现:

- **任务执行**的接口对象只有"任务"这一个——一个任务跑起来,内部步骤是黑盒;前端只关心整体进度
- **工作流**的接口对象有两个——**"工作流定义"(图)**和**"工作流执行"(图的一次运行)**。前者是可被 CRUD 的产品对象,后者才像任务一样跑;**前端不仅看整体进度,还要把"现在执行到哪个节点、节点之间传了什么变量"实时呈现给用户**

如果没用过 Dify,可以想象一下:产品后台有个"流程画板",画板上是节点和连线;运行时,**节点会被高亮、变量会在节点之间流动**——这种"图本身就是产品形态一部分"的特征,是任务执行 agent 没有的。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>工作流定义管理接口</strong></td><td align="left">创建、修改、删除、列出工作流定义</td><td align="left">管的是“图对象”,因此需要 CRUD</td></tr>
<tr><td align="left"><strong>工作流执行接口</strong></td><td align="left">用工作流 ID + 输入数据触发一次运行</td><td align="left">立刻返回执行 ID,和任务创建同类</td></tr>
<tr><td align="left"><strong>执行状态与节点进度接口</strong></td><td align="left">查询当前节点、节点输入输出和中间变量</td><td align="left">比任务进度多了“节点维度”</td></tr>
<tr><td align="left"><strong>执行结果与历史接口</strong></td><td align="left">获取某次运行的最终输出和历史记录</td><td align="left">用于复盘、追踪和复用</td></tr>
<tr><td align="left"><strong>节点中间产物访问接口</strong></td><td align="left">单独拉取某个节点产生的大 JSON、文件或图表</td><td align="left">高级产品常见,避免主事件流过重</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**和任务执行 agent 的接口对照**

<p align="center"><font face="黑体" size=4>和任务执行 agent 的接口对照</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">维度</th><th align="center">任务执行 agent</th><th align="center">工作流 agent</th></tr>
</thead>
<tbody>
<tr><td align="left">接口对象数</td><td align="left">1 个(任务)</td><td align="left">2 个(定义 + 执行)</td></tr>
<tr><td align="left">步骤可见性</td><td align="left">黑盒,只看总进度</td><td align="left">节点级别可见,要带"当前在哪个节点"</td></tr>
<tr><td align="left">CRUD 需求</td><td align="left">任务通常不复用,极少 CRUD</td><td align="left">工作流定义要被反复使用,必须 CRUD</td></tr>
<tr><td align="left">中间产物</td><td align="left">只关心最终结果</td><td align="left">节点间变量都可能要露</td></tr>
</tbody>
</table>
</div>

工作流 agent 复用了任务执行 agent 的异步任务形态,但多了**定义对象 CRUD + 节点级进度**两层。

## 5. 数据分析 agent

数据分析 agent 的最直观产品是 ChatGPT 的 Advanced Data Analysis / Claude 的 Analysis Tool / Julius AI / 各种"上传 CSV 让 AI 分析"工具。

<div align=center><font size=2 color=#999999>数据分析 agent 的产品形态(示意) — 对话区嵌入代码 / 数据表 / 图表</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-e51af1ca.jpg" width=80%></div>
<br>

产品形态上有几个一眼就能看出来的特征。

- 主对话区里用户**先上传一份 CSV / Excel 文件**,然后用自然语言提问"帮我分析一下这份销售数据,看看哪些月份波动最大,生成一份图表"
- agent 不会一次返一段答案,而是**一边写 Python 代码、一边在沙箱里跑、一边把中间结果显示给你**——代码块是可执行的 / 数据表是可滚动的 / 图表直接渲染在主对话区
- 最后产出一份**可下载的分析报告**(Excel / PDF / Notebook),用户能把整套分析结果拿走

> <font size=2>**【名词解释】<font color=red>沙箱</font>(Sandbox)** — 一个隔离的代码执行环境,通常是单独起一个进程或容器,跟 agent 主进程 / 主机文件系统 / 网络都做了隔离。agent 写出来的代码扔进沙箱里跑,即使代码 bug、死循环、想读 `/etc/passwd`,也只能在沙箱里捣腾,**碰不到外面**。数据分析 / 代码 agent 一定会用沙箱——直接让 LLM 生成的代码在主机上跑是很危险的事。</font>

我们把这一类的能力出口对照对话助手看一下,会发现多出三类——文件上传(把数据传给 agent)、代码执行 + 图表渲染(中间产物)、报告下载(最终产物)。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>聊天能力基础包</strong></td><td align="left">复用对话助手的对话能力</td><td align="left">用户通过自然语言提出分析目标</td></tr>
<tr><td align="left"><strong>文件上传接口</strong></td><td align="left">上传 CSV / Excel / Parquet 等数据文件</td><td align="left">数据分析 agent 的输入入口</td></tr>
<tr><td align="left"><strong>代码生成与执行过程接口</strong></td><td align="left">推送代码、运行输出、数据表和图表事件</td><td align="left">多事件流,比普通流式文本更结构化</td></tr>
<tr><td align="left"><strong>图表 / 中间产物访问接口</strong></td><td align="left">按 ID 拉取较大的数据表或图表</td><td align="left">事件里只放引用,不要塞大对象</td></tr>
<tr><td align="left"><strong>报告下载接口</strong></td><td align="left">下载最终 Excel / PDF / Notebook 文件</td><td align="left">分析完成后的最终产物出口</td></tr>
</tbody>
</table>
</div>

下一类我们把"文件"这个对象单独放大成一类产品——**文件处理 / 文档理解 agent**。它的核心产物不是回复,而是从一份大文件里抽出来的**结构化数据**——这件事在企业内办公场景里需求量极大,接口形态也最"密集"。

## 6. 文件处理 / 文档理解 agent

文件处理 agent 把"文档解析 + 结构化抽取"这件事单独放大成一类产品——用户传一份 PDF / Word / Excel 进去,后端跑一套重活把里头的标题 / 关键字段 / 表格抽出来,前端再围绕这份抽取结果展开问答或下载。

<div align=center><font size=2 color=#999999>文件处理 agent 的产品形态(示意) — 解析进度 + 结构化字段表 + 摘要 + 文档问答</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-2498b40b.jpg" width=80%></div>
<br>

> <font size=2>**【名词解释】<font color=red>OCR + 结构化数据</font>** — **OCR**(Optical Character Recognition,光学字符识别):把"图像形式的文字"识别成"可被搜索 / 复制 / 处理的文本字符串"。扫描版 PDF / 拍照的合同照片 / 表格图片,本身在计算机眼里就是一张图,文本框里的字打不出来——OCR 这一步先把它们变成机器可读的字符串。**结构化数据**:不是一段连续的纯文本,而是带字段名 + 字段值的有"形状"的数据,典型形态是 JSON(`{"合同方": "甲方公司", "金额": "100万", "签订日期": "2026-05-12"}`)。文件处理 agent 的核心交付物就是这种结构化数据——它从 50 页 PDF 里"识别出有哪些字段、每个字段值是什么",而不是只给一段摘要文字。</font>

最直观的产品例子:Kimi(长文档 Q&A 起家)/ ChatPDF / Notion AI 的"长文档总结" / WPS AI 文档助手 / Adobe Acrobat AI Assistant / 各种"上传 PDF 自动抽要点 / 抽合同条款"工具。

用户**上传一份 50 页的 PDF 合同 / Word 报告 / Excel 表格**,后端走一套重活——**OCR(图片版 PDF 文本化)→ 切块 → 结构化抽取(把"标题 / 章节 / 表格 / 关键字段"识别出来)→ 生成摘要**。这一套大文件能跑十几分钟,前端**必须显示进度**(已完成 30% / 当前正在处理第 15 页 / 当前步骤:OCR),完事后用户能下载抽取后的 JSON / 标注版 PDF / 摘要报告,还能在产品里基于这份文件内容继续问答(对话能力叠加上来)。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>文件上传接口</strong></td><td align="left">接收 PDF / Word / Excel / 图片等文件,返回文件 ID</td><td align="left">文件处理 agent 的输入入口</td></tr>
<tr><td align="left"><strong>文件解析接口</strong></td><td align="left">触发 OCR、切块、抽取和摘要流程</td><td align="left">返回解析任务 ID,后台异步执行</td></tr>
<tr><td align="left"><strong>解析进度接口</strong></td><td align="left">查询 OCR 中、当前页、抽取阶段等状态</td><td align="left">用阶段反馈替代假精确进度</td></tr>
<tr><td align="left"><strong>抽取结果接口</strong></td><td align="left">取标题、章节、表格、关键字段等结构化结果</td><td align="left">这是文件处理类产品的主交付物</td></tr>
<tr><td align="left"><strong>结果下载接口</strong></td><td align="left">下载 JSON、标注 PDF 或摘要报告</td><td align="left">和数据分析报告下载同类</td></tr>
<tr><td align="left"><strong>基于解析结果的问答接口</strong></td><td align="left">解析后继续围绕这份文档对话</td><td align="left">对话范围限定在单文件解析结果内</td></tr>
</tbody>
</table>
</div>

文件处理 agent 是接口形态最"密集"的一类:上传 / 异步解析 / 进度查询 / 结构化结果取回 / 文件下载 / 对话基础包**六件事全占齐**。下一类我们把出口的"混合度"再推高一点——RAG / 知识库 agent 不只是聊天 + 工具调用,还要把**检索结果作为一等公民展示给前端**(引用来源、原文片段、相关性分数都要)。

## 7. RAG / 知识库 agent

最直观的产品例子是NotebookLM(谷歌的资料库问答)/ Glean(企业知识库问答)/ ChatGPT 的 GPTs 知识库模式 / 各家公司内部基于 LangChain / LlamaIndex / LangGraph 自建的客服 / 法务 / 财务知识库。

<div align=center><font size=2 color=#999999>RAG agent 的产品形态(示意) — 知识库管理 + 带引用角标的回复 + 悬停弹出来源</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-5f72e840.jpg" width=80%></div>
<br>

产品形态上和对话助手很像——左边一栏会话列表 / 中间对话主区 / 底下输入框——但有两处明显的差别。

第一,**回复里一般会带"引用角标"**。agent 答一段话,中间夹着 `[1]` `[2]` `[3]` 这样的角标,鼠标移到角标上能弹出小框显示引用来源(文档名 + 原文片段 + 页码)。这是 RAG 类产品和纯对话最大的不同——**"答案怎么来的"必须可追溯**。

第二,产品里通常有一个**独立的"知识库管理"界面**。用户能上传文档(PDF / Word / Markdown / 网页链接)、删文档、看哪些文档已入库、看每份文档的解析状态(刚上传 / 切块中 / 已向量化 / 可被检索)、按文件名 / 标签搜索文档。

我们把这两个面孔拆开,RAG agent 比对话助手多了"文档"这个独立对象——文档既要被管理(增删改查),又要被检索(用作回复的依据)。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>聊天能力基础包</strong></td><td align="left">复用对话助手的对话能力</td><td align="left">RAG 本质是带知识来源的加强版对话助手</td></tr>
<tr><td align="left"><strong>回复与引用来源接口</strong></td><td align="left">回复里带文档片段、来源和相关性分数</td><td align="left">前端渲染引用角标和来源详情</td></tr>
<tr><td align="left"><strong>文档管理接口</strong></td><td align="left">上传、列出、删除、修改文档标签 / 名称</td><td align="left">文档是一类独立业务对象</td></tr>
<tr><td align="left"><strong>文档解析进度接口</strong></td><td align="left">查询切块、向量化、入库等异步步骤</td><td align="left">和任务执行的进度查询同类</td></tr>
<tr><td align="left"><strong>检索结果预览接口</strong></td><td align="left">在正式回答前预览相关文档片段</td><td align="left">高级产品才有;把检索能力单独暴露出来</td></tr>
</tbody>
</table>
</div>

RAG 把"文档检索"做成一类产品形态——业务对象仍然是"文档",只是用法跟前一节(文件处理)反过来:文件处理以"输出结构化结果"为终点,RAG 以"用文档片段做答案的依据"为终点。下一类我们把业务对象从"文档"换成"我们手里的代码和命令"——代码 agent。

## 8. 代码 agent

> <font size=2>**【名词解释】<font color=red>diff</font>** — 两个版本的文本/代码之间的"差异表示"。最常见的展示形态是 **<font color=green>绿底 + 号</font>表示新增的行 / <font color=red>红底 - 号</font>表示删除的行**,改动则等价于"先删后加"。Git 里用 `git diff` 看本地未提交的改动,GitHub PR 里看到的"绿红色块"就是 diff 渲染。**代码 agent 的核心产品形态就是"边推理边推送 diff 给前端"**——agent 要改文件,前端实时显示这个 diff,用户看完点确认才真正写入磁盘。本节后面提到的"diff 流 / 实时 diff"都是这个意思。</font>

<div align=center><font size=2 color=#999999>代码 agent 的产品形态(示意) — 实时 diff + 终端日志流 + 需要确认弹框</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-0aa41b25.jpg" width=80%></div>
<br>

代码 agent 偏"实时同步交互"——它和我们处在**紧密协作**的状态里,改一行 diff 就实时刷一行,跑一条命令就实时回一行命令输出。典型代表:Cursor(IDE 内嵌)/ Claude Code/Codex(终端 / CLI)/ Cline(VS Code 扩展)——agent 在 IDE 或终端里和我们一起改代码、跑测试、跑命令,大部分动作即问即答;即使有长任务,也通常以"对话内分多轮"形式表现,不像任务执行 agent 那样独立成一个任务对象。

产品形态上有几个突出特征——

- 主交互区是**代码编辑器、终端或浏览器**,不是聊天框
- agent 工作时,前端实时显示**代码改动 diff**(哪一行加 / 哪一行删 / 哪一行改)、**终端命令输出**(它在我们机器上跑了什么命令、标准输出 / 错误输出的逐行流出)
- 我们可以**中途干预**——比如 agent 跑到一半,弹一个 `需要确认: 这个命令会删除文件,是否继续?`,我们点确认才继续

这一类 agent 的接口出口有两个明显增量:<b>代码 diff / 命令日志的实时推送</b>,和<b>"需要用户确认"的双向交互</b>。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>对话 + 任务基础包</strong></td><td align="left">复用聊天能力和任务创建 / 状态 / 结果 / 取消</td><td align="left">聊天发起需求,任务承载长动作</td></tr>
<tr><td align="left"><strong>代码改动事件流接口</strong></td><td align="left">实时推送 agent 对文件做出的 diff</td><td align="left">让用户看见“改了什么”</td></tr>
<tr><td align="left"><strong>命令执行日志流接口</strong></td><td align="left">推送标准输出、错误输出和退出码</td><td align="left">远端执行,前端像看终端一样实时看</td></tr>
<tr><td align="left"><strong>需要用户确认接口</strong></td><td align="left">副作用动作前暂停,等待用户回执继续或取消</td><td align="left">工具事件流之上的双向确认</td></tr>
<tr><td align="left"><strong>工作区与命令权限接口</strong></td><td align="left">配置目录范围、命令黑白名单、应用 / 网站权限</td><td align="left">可选;企业级场景必备</td></tr>
</tbody>
</table>
</div>

下一类把"双向"再推到极致:多个 agent 在一个会话里协作,前端要同时看到几个 agent 的视角。

## 9. 多智能体协作 agent

多智能体协作类的产品例子:**Claude Sub-agents**/ **OpenAI Agents SDK**/ **Google ADK**/ **LangGraph 多 agent 编排**/ **CrewAI**(角色化多 agent 框架)。一个软件开发任务可能分给"产品经理 agent"+ "架构师 agent" + "前端 agent" + "测试 agent" 几个角色,各 agent 之间互相对话、协商方案、最后产出一份方案文档。

<div align=center><font size=2 color=#999999>多智能体协作 agent 的产品形态(示意) — 多角色分栏 + 跨 agent 消息流 + 协作过程日志</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-15b77885.jpg" width=80%></div>
<br>

产品形态上最显著的特征:**多个 agent 同时在线 + 它们之间的对话过程对用户可见**。前端通常按 agent 角色分栏(或用不同颜色的气泡区分),用户能看到"产品经理 agent 跟架构师 agent 说了什么"、"架构师 agent 跟测试 agent 怎么吵架"。在多智能体里,用户**不一定是消息的发起者**——agent A 给 agent B 发消息也是一条对话。

这一类的接口出口在前面几类基础上,关键增量是<b>"消息发送者"维度增加了</b>——不再只有"用户 → agent"和"agent → 用户"两种,还有"agent A → agent B"。前端要能区分每条消息<b>是谁说的</b>(用户 / agent A / agent B / 系统)、<b>说给谁听的</b>(整组 / 某个 agent / 全员广播)。

<p align="center"><font face="黑体" size=4>业务出口表</font></p>
<div align="center">
<table class="business-outlet-table" align="center">
<thead>
<tr><th align="center">业务出口</th><th align="center">产品层含义</th><th align="center">设计要点</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>基础能力包</strong></td><td align="left">复用聊天、任务执行和事件流接口</td><td align="left">前几类能力叠加在一起</td></tr>
<tr><td align="left"><strong>多 agent 角色配置接口</strong></td><td align="left">管理 agent 列表、角色、模型和提示词</td><td align="left">角色本身成为业务对象</td></tr>
<tr><td align="left"><strong>跨 agent 消息流接口</strong></td><td align="left">每条消息带发送方、接收方和消息性质</td><td align="left">前端按角色分栏或泳道渲染</td></tr>
<tr><td align="left"><strong>协作过程结构化日志接口</strong></td><td align="left">记录谁在什么时刻对谁说了什么</td><td align="left">支持回看、审计和导出</td></tr>
<tr><td align="left"><strong>运行控制接口</strong></td><td align="left">暂停、取消、人工接管、单 agent 状态查询</td><td align="left">多 agent 长协作里必须让用户可干预</td></tr>
</tbody>
</table>
</div>

多智能体协作 agent 的接口出口在数量上不一定比前几类多,但**每条出口的数据结构都比前面复杂一档**——消息要带"发送方 + 接收方";事件要带"agent 角色";状态要按"agent 维度"展开。

这种"多一层 agent 角色维度"的差别,直接影响下章 HTTP 事件流协议的设计——前端要按角色分栏渲染,就要先看"消息从哪个 agent 来 / 发给哪个 agent / 属于第几轮协作"这三件事。**工具调用 agent 那节单 agent 场景下设计好的事件流不能直接复用,要在协议层和数据结构层都做扩展**。

这一类产品形态出现得晚、生态还在快速演进。本节把它作为“agent 数量维度”的扩展边界:多 agent 产品至少要在消息、事件和状态里带上 agent 角色维度。

## 10. 9 类 agent 业务接口汇总

我们把上面 9 节看到的出口横向汇总一遍。下章我们会拿这里列出来的业务接口,翻译成具体的协议形态。

<p align="center"><font face="黑体" size=4>9类agent业务接口汇总</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">agent 类型</th><th align="center">核心业务出口(纯产品语言)</th></tr>
</thead>
<tbody>
<tr><td align="left">①</td><td align="left">对话助手</td><td align="left">聊天接口 / 流式回复接口 / 会话历史 / 会话管理</td></tr>
<tr><td align="left">②</td><td align="left">工具调用</td><td align="left">(对话基础包)+ 工具调用过程接口(事件流)/ 工具结果接口</td></tr>
<tr><td align="left">③</td><td align="left">任务执行</td><td align="left">(对话基础包)+ 任务创建 / 任务状态 / 任务结果 / 任务取消 / 任务步骤日志</td></tr>
<tr><td align="left">④</td><td align="left">工作流</td><td align="left">工作流管理(CRUD)/ 工作流执行 / 执行状态 + 节点进度 / 执行历史</td></tr>
<tr><td align="left">⑤</td><td align="left">数据分析</td><td align="left">(对话基础包)+ 文件上传 / 代码执行事件流 / 图表中间产物 / 报告下载</td></tr>
<tr><td align="left">⑥</td><td align="left">文件处理 / 文档理解</td><td align="left">文件上传 / 文件解析 / 解析进度 / 抽取结果 / 结果下载 / (对话基础包)</td></tr>
<tr><td align="left">⑦</td><td align="left">RAG / 知识库</td><td align="left">(对话基础包)+ 回复带引用来源 / 文档管理(CRUD)/ 文档解析进度 / 检索结果预览</td></tr>
<tr><td align="left">⑧</td><td align="left">代码</td><td align="left">(对话基础包)+ (任务能力)+ 代码 diff 事件流 / 命令日志流 / 需要用户确认事件</td></tr>
<tr><td align="left">⑨</td><td align="left">多智能体协作</td><td align="left">(对话基础包)+ 多 agent 角色配置 / 跨 agent 消息流 / 协作过程日志</td></tr>
</tbody>
</table>
</div>

上面这张表列的是**主流程业务出口**。真实 agent 产品上线时,还要叠一组**拓展出口**:这些不属于某一类型独有,而是各类产品都可能需要的增量。


<table align="center">
<thead>
<tr><th align="center">拓展出口类别</th><th align="center">产品层含义</th><th align="center">哪一类 agent 尤其重要</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>用户与权限</strong></td><td align="left">谁能调用这个接口、谁能看哪些数据</td><td align="left">全部类型</td></tr>
<tr><td align="left"><strong>资源归属与可见性</strong></td><td align="left">会话 / 文件 / 知识库属于哪个用户 / 组织,是否可分享</td><td align="left">对话助手 / RAG / 文件处理</td></tr>
<tr><td align="left"><strong>工具 / 能力开关</strong></td><td align="left">哪些工具自动执行、哪些需用户授权</td><td align="left">工具调用 / 代码 agent</td></tr>
<tr><td align="left"><strong>危险动作确认</strong></td><td align="left">删文件 / 跑高消耗操作 / 发邮件 这类副作用动作的二次确认</td><td align="left">任务执行 / 代码 agent</td></tr>
<tr><td align="left"><strong>产物列表与清理</strong></td><td align="left">历史任务产物 / 中间文件 / 临时数据的列出、保留、清理</td><td align="left">任务执行 / 数据分析 / 文件处理 / 代码 agent</td></tr>
<tr><td align="left"><strong>运行中断与人工接管</strong></td><td align="left">agent 跑到一半的暂停 / 取消 / 切人工接管</td><td align="left">任务执行 / 代码 agent / 多智能体</td></tr>
<tr><td align="left"><strong>审计日志</strong></td><td align="left">谁在什么时间调了什么接口、做了什么动作</td><td align="left">全部类型(尤其企业级)</td></tr>
</tbody>
</table>
</div>

# <center>第 2 章: 从业务接口到 HTTP 形态和 FastAPI 实现</center>

第 1 章已经把 agent 的业务出口列出来了。本章直接做两件事:先把业务出口翻译成 HTTP 形态,再用 FastAPI 写成关键代码片段。

<div align=center><font size=2 color=#999999>一个 agent 服务的嵌套结构</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-303cb43c.jpg" width=80%></div>
<br>

## 1. HTTP 形态判断卡:6 个问题就够

&emsp;&emsp;业务出口怎么落到 HTTP 形态上?需要一把共通的判断工具——这一节就是来打造这把工具。我们把判断浓缩成一张 **6 维度装配卡**:本章后面每一节业务接口,**都先拿这张卡过一遍判断 HTTP 形态,再进 FastAPI 实现**。

&emsp;&emsp;在装配之前,先回顾一下 **HTTP 请求的最小结构**——它就是装配卡要拼的"零件库"。一个 HTTP 请求由 4 个部分组成:**请求方法**(`GET` / `POST` 等动词,表示动作语义)+ **URL**(`/users/42?limit=20`,定位资源,可带 Query 参数)+ **请求头**(`Content-Type: application/json` 等元信息)+ **请求体**(一般`POST` / `PATCH` 才有,放具体数据,通常是 JSON)。后端返回的 **响应** 也有 3 件:**状态码**(`200` 成功 / `404` 找不到 / `422` 参数错 等)+ **响应头** + **响应体**。

&emsp;&emsp;**举个最小的具体例子**——前端创建一条待办,后端返新建的这条待办:

```http
# 前端发出的请求
POST /todos HTTP/1.1
Host: api.example.com
Content-Type: application/json

{"title": "买牛奶", "done": false}

# 后端返回的响应
HTTP/1.1 200 Created
Content-Type: application/json

{"id": 42, "title": "买牛奶", "done": false}
```

&emsp;&emsp;**5 个常见请求方法的语义**:`GET` 读 / `POST` 创建 / `PATCH` 部分改 / `PUT` 整体替换 / `DELETE` 删除——这就是装配卡里要选出来的"动词"零件。装配卡做的事就是:**从"方法 + URL + 请求体 + 响应类型"这一堆零件里选,拼出当前业务接口的具体 HTTP 形态**。

<p align="center"><font face="黑体" size=4>HTTP 形态 6 维度装配卡</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">判断维度</th><th align="center">走 A</th><th align="center">走 B</th></tr>
</thead>
<tbody>
<tr><td align="left">1</td><td align="left">入参字段简单还是复杂?</td><td align="left">简单字段 → <code>Path / Query</code></td><td align="left">嵌套对象 / 多字段配置 → <code>Body</code></td></tr>
<tr><td align="left">2</td><td align="left">会不会改服务端状态?</td><td align="left">只读 → <code>GET</code></td><td align="left">改:<code>POST</code> 创建 / <code>PATCH</code> 改部分 / <code>PUT</code> 整替 / <code>DELETE</code> 删</td></tr>
<tr><td align="left">3</td><td align="left">要不要边算边返?</td><td align="left">等算完一次返 → JSON</td><td align="left">边生成边返 → <code>StreamingResponse</code></td></tr>
<tr><td align="left">4</td><td align="left">(实时返回时才问)纯文本流还是多事件?</td><td align="left">纯打字机 → <code>text/plain</code></td><td align="left">区分 thinking / tool_call / result → SSE(<code>text/event-stream</code>)</td></tr>
<tr><td align="left">5</td><td align="left">涉及文件吗?</td><td align="left">上传 → <code>UploadFile + multipart/form-data</code></td><td align="left">下载 → <code>FileResponse</code></td></tr>
<tr><td align="left">6</td><td align="left">是不是长任务?</td><td align="left">短任务 → 同步 / 流式返</td><td align="left">长任务 → 拆"创建 task → 查状态 → 取结果"三段式</td></tr>
</tbody>
</table>
</div>

> &emsp;&emsp;**表里这些组件名(`Path` / `Query` / `Body` / `StreamingResponse(text/plain)` / `StreamingResponse(text/event-stream)` / `UploadFile` + `multipart/form-data` / `FileResponse` / `WebSocket` 等)都是"装配选项标签"——这里先记位置,具体语义在后续章节展开**:CRUD(§3)/ 朴素聊天 + Body(§4)/ 流式 + StreamingResponse(§5)/ Response 矩阵(§6)/ SSE(§7)/ 文件上传 + multipart(§8)/ 文件下载 + FileResponse(§10)/ 长任务三段式(§11)/ WebSocket(§12)。**这里看不懂某个术语属于正常**,跟着装配卡先建立 6 维度选型直觉即可。

&emsp;&emsp;**举两个组装例子感受一下**:

- "**只读** / **入参简单** / **一次返 JSON**" → 拼出来是 `GET /xxx + Path/Query + JSON`(典型场景:会话历史接口)
- "**创建** / **入参嵌套** / **边算边返** / **多事件**" → 拼出来是 `POST /xxx + Body + SSE`(典型场景:聊天 + 工具调用过程接口)

&emsp;&emsp;这 6 个维度只解决一个问题:接口适合什么 HTTP 形态。

## 2. 会话历史 / 任务状态接口

&emsp;&emsp;会话历史 / 任务状态接口是最简单的一类形态:前端拿一个 ID,反过来问后端"这个资源现在是什么样"。

&emsp;&emsp;**举两个例子来表达具体展现形式**:

1. **会话历史**:在 ChatGPT 左侧栏点"昨天那个会话",前端发请求带上 `session_id`,后端返这条会话的所有消息 `[{role, content, ts}, ...]`,前端渲染到对话主区。

2. **任务状态**:数据分析 agent 在跑一个大表分析(已经几分钟了),前端**每 2 秒拉一次** `GET /tasks/{task_id}/status`,后端返 `{status: 'running', progress: 0.4, current_step: '清洗缺失值'}`,前端把进度条从 30% 涨到 40%。这两件事核心结构一样:**拿 ID 反查状态**。

&emsp;&emsp;**6 维度过一遍**:

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">简单</td><td align="left">一个 ID 就够</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">只读</td><td align="left">前端只是拉信息,不改后端状态</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">单次请求拉一次状态。任务进度的"持续追踪"由前端轮询 / SSE 实现,但单次接口本身不是流式</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">单次拉一次状态本身是快的</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`GET + Path` 参数**——`GET /sessions/{session_id}/messages` / `GET /tasks/{task_id}/status`。资源 ID 走 URL Path(`session_id` / `task_id`),可选过滤参数走 Query(`?limit=20&offset=0`)。

> <font size=2>**【名词解释】<font color=red>Path 参数 / Query 参数</font>** — URL 里两种参数位置。Path 参数是 URL 路径的一部分(`/users/{user_id}` 里的 `user_id`),通常用来标识"具体哪一个资源";Query 参数挂在 `?` 之后(`/users?limit=20`),通常是过滤、分页、排序这类可选条件。FastAPI 用类型注解或 `Path(...)` / `Query(...)` 显式声明。</font>

### FastAPI 实现: 会话历史 / 任务状态接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, HTTPException, Path, Query
from typing import Annotated

# 本节新出现的写法 —— Annotated:
#   Annotated[X, Path(...)] 把「类型注解」和「参数元数据」绑在一起,
#   是 FastAPI 0.100+ 推荐写法,比 `session_id: str = Path(...)` 更友好

app = FastAPI()


# 前端请求示例:GET /sessions/abc123/messages?limit=20&offset=0
# 后端返回示例:{"session_id": "abc123", "messages": [{"role": "user", "content": "..."}, ...]}
@app.get("/sessions/{session_id}/messages")
async def get_messages(
    session_id: Annotated[str, Path(min_length=1)],
    limit: Annotated[int, Query(ge=1, le=100)] = 20,
    offset: Annotated[int, Query(ge=0)] = 0,
) -> dict:
    # 资源不存在 → HTTPException 抛出,FastAPI 自动包成 404 响应
    if session_id not in _sessions:
        raise HTTPException(status_code=404, detail="session not found")
    ...  # 业务逻辑省略


# 前端请求示例:GET /tasks/t_42/status
# 后端返回示例:{"task_id": "t_42", "status": "running", "progress": 0.4, "current_step": "清洗缺失值"}
@app.get("/tasks/{task_id}/status")
async def get_task_status(task_id: Annotated[str, Path(min_length=1)]) -> dict:
    ...  # 同样的「拿 ID 查当前状态」结构,只是 payload 字段不同

&emsp;&emsp;**前置课用法的快速激活**:`Path(min_length=1)` 给 URL 路径参数加约束(session_id 不能空字符串),`Query(ge=1, le=100)` 给查询参数加范围(分页大小 1 到 100),失败都是 FastAPI 自动返 422。`Annotated[str, Path(...)]` 这种写法把"类型注解"和"参数元数据"合在一起,是 Pydantic v2 + FastAPI 推荐的写法——比老式 `session_id: str = Path(min_length=1)` 更适合类型检查器(mypy / pyright)。

## 3. 待办 / 文档管理接口(RESTful CRUD)

> <font size=2>**【名词解释】<font color=red>RESTful CRUD</font>(REpresentational State Transfer 资源式接口设计风格)** — 把每个业务对象当成一种"资源"(`/todos`、`/sessions`、`/docs`),然后用 HTTP 动词区分动作:`GET` 读、`POST` 创建、`PATCH`(部分改)/ `PUT`(整体改)、`DELETE` 删。同一个资源路径 + 不同动词 = 不同操作,前后端看路径 + method 就知道在做什么。</font>

&emsp;&emsp;**举两个例子来表达具体展现形式**:

1. **待办应用改完成状态**:右栏点一个待办的对勾按钮 → 前端发 `PATCH /todos/42`,Body 是 `{done: true}` → 后端改 SQLite 里这条记录的 `done` 字段为 true → 返成功 → 前端把这一行划成灰色已完成。这就是"改"动作的最小完整链路。

2. **RAG agent 文档管理**:上传一份新文档 → `POST /docs`;删一份不要的文档 → `DELETE /docs/{id}`;给文档改标签 → `PATCH /docs/{id}` Body `{tags: ['法律', '合同']}`;列出所有文档 → `GET /docs`。这是一组 4 个动词全用上的 CRUD 经典样本。


<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">动作</th><th align="center">维度 2 取值</th><th align="center">HTTP 动词</th></tr>
</thead>
<tbody>
<tr><td align="left">列出 / 单条读</td><td align="left">只读</td><td align="left"><code>GET</code></td></tr>
<tr><td align="left">新建</td><td align="left">创建</td><td align="left"><code>POST</code></td></tr>
<tr><td align="left">部分修改字段(改 <code>done</code> / 改标签)</td><td align="left">部分改</td><td align="left"><code>PATCH</code></td></tr>
<tr><td align="left">整体替换</td><td align="left">整体改</td><td align="left"><code>PUT</code></td></tr>
<tr><td align="left">删除</td><td align="left">删</td><td align="left"><code>DELETE</code></td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`GET / POST / PATCH / PUT / DELETE` 五动词全用上**,按业务对象分组路由(`/todos/*`、`/docs/*`、`/sessions/*`)。

&emsp;&emsp;<font color=red>`PATCH` 和 `PUT` 的区别值得我们留心</font>:`PATCH` 是"只改我传过来的这几个字段,其他字段保持不变",`PUT` 是"用我传过来的整份对象替换原来的整份"。改一个 `done` 字段我们用 `PATCH` 更合适,因为不需要把待办的标题 / 创建时间 / 优先级再传一遍。

### FastAPI 实现: CRUD 接口

&emsp;&emsp;**先回忆 FastAPI 的路由分发机制**:同一个 URL `/items`,HTTP method 不同就是不同函数。FastAPI 按 **URL + method 的组合**决定走哪个处理函数:

<div align="center">
<table align="center">
<thead>
<tr><th align="center">请求</th><th align="center">路由分发到</th></tr>
</thead>
<tbody>
<tr><td align="left"><code>GET /items</code></td><td align="left"><code>list_items()</code></td></tr>
<tr><td align="left"><code>POST /items</code></td><td align="left"><code>create_item(body)</code></td></tr>
<tr><td align="left"><code>PATCH /items/42</code></td><td align="left"><code>patch_item(42, body)</code></td></tr>
<tr><td align="left"><code>DELETE /items/42</code></td><td align="left"><code>delete_item(42)</code></td></tr>
</tbody>
</table>
</div>

In [ ]:
from fastapi import FastAPI, HTTPException, Path
from pydantic import BaseModel, Field
from typing import Annotated

# 本节新出现的:
#   BaseModel / Field — Pydantic 请求体校验(前置课讲过,这里直接用)
#   status_code=201   — 装饰器参数,显式声明返回状态码(REST 标准:创建用 201)
#   model_dump(exclude_unset=True) — Pydantic 方法,实现「只改传入字段」语义

app = FastAPI()

&emsp;&emsp;先定义两个请求体 schema:**创建用** 和 **部分更新用** —— 这两个不能共用一个 schema,因为创建时 `title` 必填,而 PATCH 时 `title` 可以不传(代表不改)。

In [ ]:
# 创建用 schema —— title 必填
class ItemCreate(BaseModel):
    # ... 表示「必填,无默认值」;min/max_length 防空 / 防超长
    title: str = Field(..., min_length=1, max_length=500)


# 部分更新用 schema —— 每个字段都是 Optional
class ItemPatch(BaseModel):
    # title 和 done 都可不传;不传 = 不改这个字段
    title: str | None = Field(default=None, max_length=500)
    done: bool | None = None

&emsp;&emsp;接下来分 4 个路由 —— 列表、创建、更新、删除 —— 逐个看。前两个是常规操作:

In [ ]:
# GET /items —— 列表接口,直接返一个 list
# 前端请求示例:GET /items
# 后端返回示例:[{"item_id": 1, "title": "买菜", "done": false}, {"item_id": 2, "title": "写报告", "done": true}]
@app.get("/items")
async def list_items() -> list[dict]:
    ...  # 业务逻辑省略,demo 只看接口形态


# POST /items —— 创建接口
# status_code=201 显式声明「创建成功返 201」(REST 标准,而非默认 200)
# 前端请求示例:POST /items   Body: {"title": "买菜"}
# 后端返回示例:{"item_id": 3, "title": "买菜", "done": false}
@app.post("/items", status_code=201)
async def create_item(body: ItemCreate) -> dict:
    ...

&emsp;&emsp;**PATCH 是这一节的核心知识点** —— 真正实现「只改传入字段、其他不动」的关键就在最后一行的 `model_dump(exclude_unset=True)`:

In [ ]:
# PATCH /items/{item_id} —— 部分更新接口
# 前端请求示例:PATCH /items/3   Body: {"done": true}    ← 只改 done,title 不动
# 后端返回示例:{"item_id": 3, "title": "买菜", "done": true}
@app.patch("/items/{item_id}")
async def patch_item(
    item_id: Annotated[int, Path(ge=1)],  # Path 校验:必须是 >=1 的整数
    body: ItemPatch,
) -> dict:
    # 404 用 HTTPException 抛出 —— FastAPI 会包成标准错误响应
    if item_id not in _store:
        raise HTTPException(status_code=404, detail="item not found")

    # exclude_unset=True 只打包前端实际传了的字段 → 实现「只改传入字段」语义
    return _store.patch(item_id, body.model_dump(exclude_unset=True))

In [ ]:
# DELETE /items/{item_id} —— 删除接口
# 前端请求示例:DELETE /items/3
# 后端返回示例:{"deleted": 3}
@app.delete("/items/{item_id}")
async def delete_item(item_id: Annotated[int, Path(ge=1)]) -> dict:
    ...
    return {"deleted": item_id}

&emsp;&emsp;**实际工程里 PATCH 比 PUT 用得多得多**(整体替换的场景很少),这一节我们把五动词都列了一遍是为了**保留完整方法论**,工程里按业务需要裁。

## 4. 聊天接口（非流式聊天接口）

&emsp;&emsp;非流式聊天接口对外提供的能力很直白:用户敲完一条消息按回车,前端把这条消息 + 之前的对话历史一起送到后端,后端拿到上下文喂给 agent,等 agent 推理完返一段完整回复。它是朴素的“一发一收”:不带流式、不带事件分类,中间用户什么也看不到。

&emsp;&emsp;**展现形式**:打开 ChatGPT / Claude / 智谱清言网页版,在输入框里敲"明天提醒我交报告"按回车——前端不是只把这条消息送到后端,而是**把当前会话里此前的所有消息打包成一个 `messages` 数组**(`[{role: 'user', content: '...'}, {role: 'assistant', content: '...'}, ...]`),连同 `session_id`、`stream`(是否流式)、`temperature`、`model` 等多个 agent 配置字段一起发出去。后端拿到这段嵌套数据,喂给 LangChain agent 推理,返一条完整回复。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left"><strong>复杂</strong></td><td align="left">不是 1-2 个字段,而是嵌套的 <code>messages: list[Message]</code> + 多个 agent 配置字段。这种结构没法塞进 URL Query,必须走请求体</td></tr>
<tr><td align="left">维度 2(是否改状态)</td><td align="left"><strong>改</strong></td><td align="left">这一次发消息会"在数据库 / 内存里多创建一条对话记录",属于创建操作</td></tr>
<tr><td align="left">维度 3(是否实时)</td><td align="left">本节先按<strong>否</strong>处理</td><td align="left">—</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">—</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`POST /chat` + `Body`(JSON)+ `Pydantic` 校验**。`POST` 来自维度 2(创建),`Body` 来自维度 1(复杂入参),`Pydantic` 来自“嵌套结构需要声明式校验”。

### FastAPI 实现: 聊天接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI()


class Message(BaseModel):
    role: Literal["user", "assistant", "system"]
    content: str = Field(..., min_length=1)


class ChatRequest(BaseModel):
    # agent 入参典型形态:嵌套 list[Message] + 配置字段
    messages: list[Message] = Field(..., min_length=1)
    session_id: str | None = None
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)


# 前端请求示例:POST /chat
#   Body: {"messages": [{"role": "user", "content": "你好"}], "session_id": "s_001", "temperature": 0.7}
# 后端返回示例:{"reply": "你好,有什么可以帮你的?", "session_id": "s_001"}
@app.post("/chat")
async def chat(req: ChatRequest) -> dict:
    # 到这里时 FastAPI 已经完成 Body 解析 + 嵌套校验
    ...

> <font size=2>**【名词解释】<font color=red>Pydantic 嵌套 schema</font>** — Pydantic 的 `BaseModel` 字段类型可以是另一个 `BaseModel`(`messages: list[Message]`),FastAPI 会递归解析嵌套层级,自动按外层 + 内层一起校验。前置课只演示过单层 BaseModel(纯字段),agent 场景里嵌套是必须的。</font>

&emsp;&emsp;**重点升级:agent 入参嵌套 schema**。前置课讲过 `BaseModel` 的"单层字段"用法——一个 `class UserIn(BaseModel): name: str; age: int` 收下用户名和年龄。但 agent 场景的请求体没这么扁平,**至少有两层**:外层是 `ChatRequest`(整个请求),内层是 `list[Message]`(对话历史)。每条 `Message` 又有自己的字段约束(`role` 必须是三选一,`content` 不能空)。FastAPI 拿到请求 JSON 后,会**递归**按外层 + 内层一起校验——任意一层任意一条不合格,整个请求直接 422,**不进我们的路由函数**。

&emsp;&emsp;比如前端发 `{"messages":[{"role":"reviewer","content":"hi"}]}`,`role` 不在 `Literal["user","assistant","system"]` 三个值里,FastAPI 直接返:

```json
{
  "detail": [
    {
      "type": "literal_error",
      "loc": ["body", "messages", 0, "role"],
      "msg": "Input should be 'user', 'assistant' or 'system'",
      "input": "reviewer"
    }
  ]
}
```

&emsp;&emsp;`loc` 字段精确到 `messages[0].role`——前端拿到这个就知道是第一条消息的 role 字段错了。**这是嵌套 schema 带给我们的免费红利**:不用手写一行校验逻辑,错误定位还能精确到嵌套字段。

&emsp;&emsp;<font color=red>为什么不用 `dict` 接请求体省事?</font>把 `chat` 函数签名改成 `async def chat(req: dict)`,FastAPI 也能跑——但**我们就放弃了所有校验**,前端传 `{"messages": "hi"}`(字符串而不是数组)、传 `{"role": null}` 这些畸形数据,代码会跑到 `req["messages"][-1].content` 这一行才崩,500 错误响应给前端,前端根本不知道哪儿错了。**嵌套 schema 的本质是"把校验代价从运行时挪到声明时"**——我们多写几行 `class Message` 的代价,换来前端能拿到结构化的错误定位。

&emsp;&emsp;**对接真实 agent 时的小幅扩展**:上面 `chat` 函数里返的是 mock 回复。真实项目里这一段会被替换成 `await agent.ainvoke({"messages": lc_messages})`——把 Pydantic 的 `list[Message]` 转成 agent 框架(LangChain / LangGraph 等)能吃的消息格式,喂给 agent 推理,等推理完拿回复。同步 `/chat` 适合短回复;长回复 / 工具调用过程要展示给前端时,通常走下一节的流式接口。

## 5. 聊天流式接口

&emsp;&emsp;前面我们讲了"一发一收"的聊天接口——agent 推理完整一段才返。但实际产品里,用户期待的是 ChatGPT 那种"字一个一个蹦出来"的打字机效果,这就是流式回复接口要做的事:同一件事(聊天),只是返回方式变了。前端不等 agent 推理完整一段才显示,而是**一边推一边把吐出来的字实时显示在屏幕上**。

&emsp;&emsp;**展现形式**:ChatGPT 网页版回答时,字一个一个蹦出来,光标在每个字后面跳一下——这就是流式回复。如果我们用朴素聊天接口实现这个效果,前端只能"等到后端把整段生成完一次性显示",用户看到的是空白等几秒钟,然后突然整段字"啪"地刷出来。<font color=red>体验上的差别非常明显:等几秒 vs 立刻看到第一个字蹦出来,后者明显用户体验更好</font>。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">复杂</td><td align="left">同非流式聊天接口</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">创建</td><td align="left">同非流式聊天接口</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left"><strong>是</strong></td><td align="left">这是本节核心差异。token 级流式,前端立刻看到第一个字</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left"><strong>否</strong></td><td align="left">流里只有纯文本,前端不需要区分事件类型。这条接口和 SSE 接口最关键的分水岭就在这里</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">—</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`POST /chat/stream` + `StreamingResponse(media_type="text/plain")`**。和朴素聊天接口的区别只在响应类型:请求体仍然是嵌套 `messages`,但响应不是返一整段 JSON,而是逐段 yield 字符块。

### FastAPI 实现: 聊天流式接口

&emsp;&emsp;**FastAPI 关键代码片段(接 agent 流)**:

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

app = FastAPI()


class ChatRequest(BaseModel):
    session_id: str = Field(..., min_length=1)
    message: str = Field(..., min_length=1, max_length=4000)


# 前端请求示例:POST /chat/stream   Body: {"session_id": "s_001", "message": "讲个笑话"}
# 后端返回示例:text/plain 流式分块 ——
#   chunk 1: "好"
#   chunk 2: "的,"
#   chunk 3: "有"
#   chunk 4: "一只"
#   chunk 5: "..."     ← 一直推到 LLM 输出结束,连接才关
@app.post("/chat/stream")
async def chat_stream(req: ChatRequest) -> StreamingResponse:
    async def gen():
        # 读 agent.astream,把 token 文本 yield 给前端
        async for chunk in agent.astream({"messages": [{"role": "user", "content": req.message}]}):
            text = getattr(chunk, "content", "")
            if text:
                yield text

    return StreamingResponse(
        gen(),
        media_type="text/plain",
        headers={"X-Accel-Buffering": "no"},  # 防 nginx 把流攒成一段才送
    )

> <font size=2>**【名词解释】<font color=red>astream</font>(LangChain 异步流式接口)** — LangChain 1.x agent 对象提供的异步流式方法。`agent.astream({"messages": [...]})` 返一个 async iterator,每次迭代拿到一个 `AIMessageChunk` 对象,`.content` 字段是这一刻吐出来的 token 文本(可能是单字、也可能是几个字一组)。普通 LLM 直连用 `llm.astream(...)`,agent 包装后用 `agent.astream(...)`,接口形态完全一致。</font>

&emsp;&emsp;**重点升级:对接 LangChain agent.astream 跟前置课直连 LLM 的两处差别**——

- **输入格式不同**:`llm.astream("你好")` 收字符串,`agent.astream({"messages": [...]})` 收 dict,因为 agent 内部要管理对话上下文,不是无状态的单 prompt。
- **chunk 对象不同**:`llm.astream` 直接 yield 字符串,`agent.astream` yield 的是 `AIMessageChunk` 对象,要用 `.content` 取文本。多模态场景下 `.content` 可能是 `list`(混合文本 / 图片块),取文本要再过一遍 `isinstance(item, dict) and item.get("type") == "text"`。

&emsp;&emsp;**前置课流式 → agent 流式的代码 diff**(只看核心 generator 体):

In [ ]:
# 前置课流式(直连 LLM)
async def gen():
    async for chunk in llm.astream("你好"):
        yield chunk        # chunk 直接是字符串

# 本节升级(接 agent)
async def gen():
    async for chunk in agent.astream({"messages": lc_messages}):
        text = getattr(chunk, "content", "")
        if text:
            yield text     # chunk 是 AIMessageChunk,要 .content

&emsp;&emsp;**真实项目里的两处常见扩展**:① **history 长度控制** —— LangGraph state 历史通常限制到最近 20-50 条,防 context 超长;② **要拿工具调用过程** —— 用 `agent.astream_events(version="v2")` 代替 `agent.astream`,能拿到 token 流 + 工具开始 + 工具结束等框架事件(API 细节见 §7 SSE 那一节)。纯文本流式只挑文本 token,**不适合展示完整工具调用过程**——那也是 §7 的活。

&emsp;&emsp;`X-Accel-Buffering: no` 是给 nginx 的“别 buffer 我”信号,流式 endpoint 建议带上。

## 6. 工具结果接口

&emsp;&emsp;agent 调一个工具(查天气 / 查日历 / 搜知识库),工具返回的结构化结果要传给前端,**前端按字段渲染卡片**——不是把工具结果当一段文本贴一行,而是显式拆字段(图标 + 温度 + 湿度三栏渲到天气卡片上)。这就是工具结果接口要做的事。

&emsp;&emsp;**展现形式**:agent 调"查天气"工具返 `{city: '北京', temperature: 18, condition: '晴', humidity: '45%'}`,前端拿到 JSON 后按字段渲染——左边一个晴天图标、右上角大字"18°C"、下方三栏"晴 / 湿度 45% / 北京"。这是工具调用 agent 里常见的一类业务接口,**我们要把它和工具调用过程接口区分开**:过程接口是"实时推送 agent 正在调什么工具、刚才工具返了什么"(过程层,事件流);本节是"前端单独拉某个工具的结果详情"(结果层,普通 JSON 返回)。一个是事件流,一个是请求-响应。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">简单</td><td align="left">通常我们只需要一个 <code>tool_call_id</code> 或 <code>result_id</code></td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">只读</td><td align="left">—</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">一次性返完</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">—</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`GET /tool_results/{id}` + 返 JSON**。在 FastAPI 里直接 `return dict` 就会自动转 JSON,显式形式是 `JSONResponse(content=...)`。

### FastAPI 实现: 工具结果接口

&emsp;&emsp;**先看 JSONResponse 这条主线接口怎么写**:

In [ ]:
from fastapi import FastAPI, HTTPException

app = FastAPI()


# 前端请求示例:GET /tool_results/r_abc123
# 后端返回示例:{"tool": "weather", "city": "北京", "temperature": 18, "condition": "晴"}
@app.get("/tool_results/{result_id}")
async def get_tool_result(result_id: str) -> dict:
    # 工具结果已是结构化数据,直接 return dict,FastAPI 自动转 JSON
    result = _store.get(result_id)
    if result is None:
        raise HTTPException(status_code=404, detail="tool result not found")
    return result

&emsp;&emsp;`return dict` 和 `return JSONResponse(content=...)` 在功能上等价——FastAPI 拿到 dict 自动转 JSON 写到响应体。<b>什么时候需要显式 `JSONResponse`?</b>当我们想自定义 `status_code` / 响应头 / 编码细节(`ensure_ascii=False` 保中文)时,就要显式构造 `JSONResponse` 对象。

&emsp;&emsp;**FastAPI最常见 6 类 Response 速查表**:

<p align="center"><font face="黑体" size=4>FastAPI最常见 6 类 Response</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">Response 子类</th><th align="center">media_type</th><th align="center">用来返什么</th><th align="center">典型场景</th></tr>
</thead>
<tbody>
<tr><td align="left"><code>JSONResponse</code></td><td align="left"><code>application/json</code></td><td align="left">结构化字典 / 列表</td><td align="left">工具结果 / 业务对象 / 默认形态</td></tr>
<tr><td align="left"><code>PlainTextResponse</code></td><td align="left"><code>text/plain</code></td><td align="left">纯文本字符串</td><td align="left">健康检查 <code>/health</code> 返 <code>"ok"</code> / 调试 endpoint</td></tr>
<tr><td align="left"><code>HTMLResponse</code></td><td align="left"><code>text/html</code></td><td align="left">HTML 字符串</td><td align="left">偶尔返服务端渲染的 HTML 片段 / 不上 Jinja 时</td></tr>
<tr><td align="left"><code>RedirectResponse</code></td><td align="left"><code>(无 body, 状态码 307/302)</code></td><td align="left">跳转 URL</td><td align="left">登录成功跳首页 / 短链接服务</td></tr>
<tr><td align="left"><code>FileResponse</code></td><td align="left">按扩展名自动</td><td align="left">磁盘文件流</td><td align="left">报告下载 / 导出 JSON / 静态资源</td></tr>
<tr><td align="left"><code>StreamingResponse</code></td><td align="left">自定义,常见 <code>text/plain</code> / <code>text/event-stream</code></td><td align="left">流式生成的数据</td><td align="left">打字机效果 / SSE 多事件</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**每类看一个关键返回写法**:

In [ ]:
from fastapi.responses import (
    JSONResponse, PlainTextResponse, HTMLResponse,
    RedirectResponse, FileResponse, StreamingResponse,
)


# 后端返回示例:
#   HTTP/1.1 200 OK
#   Content-Type: application/json
#   {"tool": "weather", "temperature": 18}
async def json_case() -> JSONResponse:
    return JSONResponse({"tool": "weather", "temperature": 18})


# 后端返回示例:
#   HTTP/1.1 200 OK
#   Content-Type: text/plain; charset=utf-8
#   ok
async def text_case() -> PlainTextResponse:
    # 健康检查 / 调试 endpoint 常用,前端拿到的就是裸字符串
    return PlainTextResponse("ok")


# 后端返回示例:
#   HTTP/1.1 200 OK
#   Content-Type: text/html; charset=utf-8
#   <h1>FastAPI 后端就绪</h1>
async def html_case() -> HTMLResponse:
    # 偶尔返服务端渲染的小片段(完整全栈项目通常上 Jinja2)
    return HTMLResponse("<h1>FastAPI 后端就绪</h1>")


# 后端返回示例:
#   HTTP/1.1 307 Temporary Redirect
#   Location: /dashboard
#   (无 body — 浏览器自动跳到 Location 指向的地址)
async def redirect_case() -> RedirectResponse:
    # 登录成功跳首页 / 短链接展开,浏览器自动跟随
    return RedirectResponse(url="/dashboard")


# 后端返回示例:
#   HTTP/1.1 200 OK
#   Content-Type: application/json
#   Content-Disposition: attachment; filename="report.json"
#   <文件二进制内容>   ← 浏览器自动弹下载弹窗
async def file_case() -> FileResponse:
    # 文件下载不要把 bytes 塞进 JSON,直接用 FileResponse。
    return FileResponse(path="exports/report.json", filename="report.json")


# 后端返回示例:
#   HTTP/1.1 200 OK
#   Content-Type: text/plain; charset=utf-8
#   Transfer-Encoding: chunked
#   chunk 1: "好"
#   chunk 2: "的,"
#   chunk 3: "有"
#   chunk 4: "..."   ← 流式分块,前端 ReadableStream 边收边显示
async def stream_case() -> StreamingResponse:
    async def gen():
        async for token in agent.astream(...):
            yield token.content

    return StreamingResponse(gen(), media_type="text/plain")


&emsp;&emsp;<font color=red>选错 Response 类型,前端拿不到目标效果</font>。最常见的坑有三种——

- **用 `JSONResponse` 返文件**:把文件 bytes 塞进 dict 的某个字段返出去,文件体积乘个 4/3(base64 编码膨胀)+ 浏览器不会触发下载弹窗。应该用 `FileResponse`。
- **用 `JSONResponse` 返打字机流**:agent 推完整一段才返,前端等几秒看到整段字。应该用 `StreamingResponse(media_type="text/plain")`。
- **用 `StreamingResponse(text/plain)` 返工具调用过程**:前端拿到一段纯字符,分不清哪段是 thinking 哪段是 tool_result。应该用 `StreamingResponse(media_type="text/event-stream")`,也就是 SSE。

&emsp;&emsp;**5 段式拓展:这 6 类够用吗?**

- **遇到什么问题**:同一个 endpoint 要根据业务返不同形态(JSON / 流式 / 文件 / 重定向),用错就拿不到目标效果。
- **本课怎么选**:业务数据用 `JSONResponse`(默认 dict 也行),工具结果用 `JSONResponse`,对话流用 `StreamingResponse(text/plain)`,事件流用 `StreamingResponse(text/event-stream)`,文件下载用 `FileResponse`。
- **为什么这样选**:`JSONResponse` 跟前端 JSON 生态对齐,`StreamingResponse` 两种 media_type 覆盖普通流和 SSE,`FileResponse` 自动处理大文件分块。
- **其他方案什么时候用**:`PlainTextResponse` 给 `/health` / 调试 endpoint;`HTMLResponse` 偶尔返服务端渲染的小片段(全栈大项目通常用 Jinja2);`RedirectResponse` 用在登录跳转 / 短链接服务。

&emsp;&emsp;**实际项目里通常只用 3 类左右**:`JSONResponse`(大多数接口)/ `FileResponse`(导出下载)/ `StreamingResponse`(流式 / SSE)。`PlainText` / `HTML` / `Redirect` 用得少,按需取用即可。

## 7. 工具调用过程接口 + 日志流接口

&emsp;&emsp;前面普通流式接口推的是纯文本,前端拿到啥就贴啥;工具结果接口讲的是"事后按 ID 拿最终 JSON"。但工具调用 agent 不只要拿最终结果,还要让前端**实时看到 agent 在干什么**——"它正在想"/"它正在调什么工具"/"工具返了什么"。这些信息塞在一段纯文本流里前端没法识别,我们需要的是"**带名字的多事件**"协议——这就是 SSE 的用武之地。本节我们看两个看起来完全不同的业务接口——**工具调用过程接口**和**日志流接口**——一起讲,**因为它们在协议层是同一个需求**:服务端单向推送 + 多种事件类型 + 前端按类型分发渲染。

> <font size=2>**【名词解释】<font color=red>SSE</font>(Server-Sent Events,服务端推送事件)** — 一种基于 HTTP 长连接的单向推送协议,服务端可以一段段把"带名字的事件"推给浏览器。每个事件**可有** `event:` 字段(**可选**,默认事件名 `message`) + `data:` 字段(必须,放事件载荷,通常是 JSON 字符串),前端用 `EventSource` API 按事件类型订阅。它和普通流式(`text/plain`)的关键差别:**普通流式只推纯文本,SSE 推带名字的结构化事件**。SSE 协议本身定义在 WHATWG HTML 标准里,FastAPI 通过 `StreamingResponse(media_type="text/event-stream")` 来实现。</font>

### 典型场景 A:工具调用过程接口

&emsp;&emsp;一边吐答案、一边告诉前端"agent 现在在想什么 / 调什么工具 / 工具返了什么",前端按事件类型分别渲染——这是工具调用 agent 区别于纯对话最关键的能力出口。

&emsp;&emsp;**展现形式**:打开 Claude / ChatGPT 高级模式,问"帮我查一下北京明天的天气"。主对话区开始一边吐字"明天北京……",**同时在旁边/上方实时跳出一行灰色提示** `正在搜索:北京天气`(这是 `thinking` 事件);紧接着弹出一个可折叠卡片显示工具返回结果 `{temp: 18, condition: '晴'}`(这是 `tool_result` 事件);最后 agent 继续把答案文字流回主对话区。**前端按 `thinking` / `tool_call` / `tool_result` 不同事件类型走不同 UI 块**——灰色行渲思考、卡片渲工具结果、气泡渲最终答案,各管各的。

### 典型场景 B:日志流接口

&emsp;&emsp;**agent 在远端跑一个会持续输出日志的命令**(`npm install` / `pytest` / 模型训练 / 大数据查询),把命令的 `stdout` / `stderr` 逐行实时推给前端,前端**像 GitHub Actions / Vercel 部署面板那种黑底日志窗口一样,日志一行一行向下滚动出来**。

&emsp;&emsp;**展现形式**:打开 Cursor / Devin 这类代码 agent,告诉它"帮我把这个 JS 项目改造成 TypeScript",agent 执行到 `pip install -r requirements.txt` 这一步——前端右侧面板黑底窗口里逐行刷出 `Collecting fastapi==0.136.1...`、`Downloading fastapi-0.136.1-py3-none-any.whl...`、`Successfully installed fastapi-0.136.1`。**和你本地终端看到的 pip 输出一模一样**,只不过是远端跑、前端实时看。如果在 `pytest` 跑的过程中某个测试失败,`stderr` 那一行还要标红;命令跑完后,前端要拿到 `exit_code` 决定整体标"成功"还是"失败"。

&emsp;&emsp;<b>为什么这两类都不能用普通流式?</b>因为日志流要区分多种事件——`stdout` / `stderr` / `command_start` / `command_end` / `exit_code`。`stderr` 在前端可能要标红,`exit_code` 非零要标失败——这些信息塞在一段纯文本流里前端没法识别。工具调用过程同理——`thinking` / `tool_call` / `tool_result` / `done` 各自要走不同 UI 块。

&emsp;&emsp;**它俩还是嵌套关系——不是平级的"两类场景"。**真实代码 agent(Cursor / Devin)跑一条命令的全程,两层 SSE 事件同时发生:**外层**是 agent 的工具调用过程(语义层,`tool_call` / `tool_result`),**内层**是 shell 跑命令产的日志流(进程层,`stdout` / `exit_code`)。我们以 agent 跑 `pip install -r requirements.txt` 这一步为例:

```text
agent 视角(工具调用过程层 SSE):
  ├─ tool_call:    {"tool": "shell.exec", "args": {"cmd": "pip install -r ..."}}
  │
  │   shell 视角(日志流层 SSE — 嵌在这次 tool_call 内部):
  │     ├─ command_start: {"cmd": "pip install ..."}
  │     ├─ stdout:        "Collecting fastapi==0.136.1..."
  │     ├─ stdout:        "Downloading fastapi-0.136.1-py3-none-any.whl..."
  │     ├─ stdout:        "Successfully installed fastapi-0.136.1"
  │     └─ exit_code:     0
  │
  └─ tool_result:  {"tool": "shell.exec", "output": "installed", "exit_code": 0}
```

&emsp;&emsp;**对应到后端工程实现**:Cursor 这种代码 agent 后端通常**开两条 SSE**(一条给 agent 推理面板 / 一条给终端日志窗口),也可以**一条 SSE 混合两类事件**让前端按 `event:` 名字分发到不同 UI 块。两种做法都有人用,选哪个看团队工程偏好。本节代码示例只覆盖工具调用过程那一层,日志流那一层的 endpoint 设计原理完全一样——只是把 `event:` 名字换成 `stdout` / `stderr` / `exit_code`。

&emsp;&emsp;**两种场景的共同点**:都需要服务端单向推送 + 带名字的多事件(不只纯文本)+ 前端按事件类型分发。我们把这套需求合起来,自然推出 SSE。这也是为什么 LangChain 1.x 的 **`astream_events(version="v2")`** 接口天然产出带 `event` 字段的事件流——它就是为了对接 SSE 设计的,下面的 FastAPI 实现会用它做 agent 接 SSE 的标准实现。

&emsp;&emsp;SSE 的协议格式很简单:`event: X\ndata: Y\n\n`。本节重点是先确定“带名字的多事件流”要走 SSE。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">复杂</td><td align="left">嵌套消息 + agent 配置</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">创建</td><td align="left">发起一次 agent 推理</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left"><strong>是</strong></td><td align="left">agent 边推理边推事件</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left"><strong>是</strong></td><td align="left">这是本节和普通流式接口最大的差别。事件分 <code>thinking</code> / <code>tool_call</code> / <code>tool_result</code> / <code>done</code> 等多类</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">单次推理短</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`StreamingResponse(media_type="text/event-stream")` + 业务事件协议**。每个事件带 `event:` 名字 + `data:` JSON 载荷;入口路由走 GET 还是 POST,**看输入复杂度,分两档**:

<p align="center"><font face="黑体" size=4>SSE 接口入口两档</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">档位</th><th align="center">后端入口</th><th align="center">前端订阅方式</th><th align="center">适用场景</th></tr>
</thead>
<tbody>
<tr><td align="left">简单输入档</td><td align="left"><code>GET /chat/events</code> + Query 参数</td><td align="left"><code>new EventSource(url)</code>(浏览器原生)</td><td align="left">几个字段塞 Query 够用;<strong>本节代码走这一档</strong></td></tr>
<tr><td align="left">复杂输入档</td><td align="left"><code>POST /agent/stream</code> + Body(嵌套 <code>messages: list[Message]</code> + 配置)</td><td align="left"><code>fetch + ReadableStream</code> 手 parse SSE 报文(生产用 <code>@microsoft/fetch-event-source</code> 库)</td><td align="left">Agent 实战常见;EventSource 规范限制只支持 GET,装不下大 body</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;两档**SSE 报文格式完全一样**——`event: 名字\ndata: JSON\n\n`,只是"前端怎么发起请求"和"怎么收响应"不同。SSE 本身是 `StreamingResponse` 的一种特化用法——同样是 yield 数据,但 yield 出来的不是纯字符,而是符合 SSE 格式的 `event: thinking\ndata: {...}\n\n` 这种结构化文本。本节代码先用简单档把 SSE 报文机制吃透,后面会给出复杂档的最小切换 diff。

&emsp;&emsp;**普通流式 vs SSE 多事件**对比:同样是流式,两者推的东西完全不同——普通流式推一段纯文本字流,前端只能把字贴一行;SSE 推一连串带名字的事件,前端能按事件类型分发到不同 UI 块。我们用一张图把这个对比立清。

<div align=center><font size=2 color=#999999>普通流式接口 vs SSE 多事件接口 — 前端拿到的内容形态完全不同</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-9930ef7d.jpg" width=80%></div>
<br>

&emsp;&emsp;<font color=red>SSE 有一个实战坑</font>:经过反向代理时如果配置不当会被 buffer 住——前端等几十秒一次性收到全部内容,流式效果消失。源 endpoint 要带 `X-Accel-Buffering: no`,网关也要关闭 buffering / cache / gzip。

### FastAPI 实现: SSE + 业务事件协议

&emsp;&emsp;<b>为什么不能直接用 普通流式实现?</b>我们看一个反例——用普通流式推一个工具调用过程:

```python
    # 反例:用 text/plain 流推 thinking + tool_result + answer
    async def gen():
        yield "[thinking] 正在搜索北京天气\n"
        yield "[tool_call] get_weather('北京')\n"
        yield "[tool_result] {temp: 18, condition: '晴'}\n"
        yield "明天北京晴,18 度,适合出门。\n"
```

&emsp;&emsp;前端拿到这一段纯文本,<b>它怎么区分这一行是 thinking 还是答案?</b>得自己写一段 `if line.startswith("[thinking]")` 的字符串解析——脆弱、不结构化、有 `[` 字面量在内容里时就崩。我们需要的不是"自己发明一套文本协议",而是 <b>HTTP 已经有的标准多事件协议</b>——SSE。

> <font size=2>**【名词解释】<font color=red>astream_events(version="v2")</font>** — LangChain 1.x 给 agent 提供的"全事件流"接口。普通 `astream` 只 yield token chunk,`astream_events` yield 框架所有事件——`on_chat_model_start` / `on_chat_model_stream`(token 流)/ `on_tool_start`(工具开始调) / `on_tool_end`(工具返结果) / `on_chain_start` / `on_chain_end` 等几十种。**每个事件有 5 个字段**:`event`(事件名)+ `name`(节点名)+ `data`(载荷)+ `run_id`(本次运行的唯一 ID)+ `parent_ids`(嵌套调用的父节点 ID 列表,v2 相对 v1 新增,用于追踪 agent 内部嵌套调用的层级)。**`data.chunk.content` 两种形态**:简单文本流场景下是 `str`(直接 yield 一段字);多模态(图 + 文 / 工具调用的中间结果)场景下是 `list[dict]`,每条是 `{type: "text"/"tool_use", ...}`——代码 `translate` 函数已处理两种形态。我们要把这些**框架事件**翻译成业务关心的 5-7 种**业务事件**推给前端。</font>

> <font size=2>**【名词解释】<font color=red>EventSource</font>(浏览器 SSE 客户端 API)** — 浏览器原生 API,专门消费 SSE 流。`const es = new EventSource('/chat/events?...')` 建连接,然后 `es.addEventListener('thinking', e => ...)` 按事件名订阅。**两个关键限制**:① 只支持 `GET`(不能 POST,所以参数走 Query)② 不能加自定义请求头(自定义鉴权 header 不行,鉴权要走 Query 兜底)。</font>

&emsp;&emsp;**SSE 返回格式**:每条事件三件套——

```text
event: thinking
data: {"text": "正在搜索北京天气"}

event: tool_result
data: {"tool": "get_weather", "output": {"temp": 18, "condition": "晴"}}
id: 42
```

&emsp;&emsp;**三个细节缺一不可**:① `event:` 字段是事件名(可选,缺省叫 `message`);② `data:` 字段是事件载荷,**只能单行**(JSON 不能带裸换行,序列化时用 `separators=(",", ":")` 压扁);③ 每条事件**末尾必须空一行**(`\n\n`)告诉浏览器"这条结束了"——缺一个换行整个流就崩。

&emsp;&emsp;**FastAPI 关键代码片段**:我们先实现一个 SSE 工具函数 + 一条最小 SSE endpoint:

In [ ]:
import json
from typing import Any


# 把一次"业务事件"打包成符合 SSE 协议的报文字符串(三件套见前面"三个细节缺一不可")。
#
# 输入示例:
#   sse_pack("tool_call", {"tool": "get_weather", "args": {"city": "北京"}}, id_=42)
#
# 输出报文(返回的就是这一段字符串,前端按 \n\n 切块、再按 event: / data: / id: 取字段):
#   event: tool_call
#   data: {"tool":"get_weather","args":{"city":"北京"}}
#   id: 42
#   <空行>
def sse_pack(event: str, data: dict[str, Any] | None = None, id_: int | None = None) -> str:
    # event/data/id 三行 + 末尾空行 = 一条 SSE 事件
    payload = json.dumps(data or {}, ensure_ascii=False, separators=(",", ":"))
    lines = [f"event: {event}", f"data: {payload}"]
    if id_ is not None:
        lines.append(f"id: {id_}")
    return "\n".join(lines) + "\n\n"


&emsp;&emsp;**为什么不直接把 LangChain 的框架事件名透传给前端?**因为直接透传 `on_chat_model_stream` / `on_tool_start` 这套字符串虽然能跑通,但**把前端跟 LangChain 的内部命名锁死了**——LangChain 升级到 2.x、改了事件名,前端就要跟着改。**我们需要的是一个翻译层**:把 LangChain 的几十种框架事件,映射成业务关心的 7-8 种**业务事件**(消息流 / 工具调用 / 工具结果 / 思考态 / 完成 / 失败 / 需确认)推给前端。前端只对接业务事件名,框架升级不影响。

<p align="center"><font face="黑体" size=4>业务事件协议表</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">业务事件名</th><th align="center">触发源</th><th align="center">payload 字段</th><th align="center">前端 UI 含义</th></tr>
</thead>
<tbody>
<tr><td align="left"><code>run_started</code></td><td align="left">路由开头 yield</td><td align="left"><code>{session_id}</code></td><td align="left">显示"AI 思考中..." loader</td></tr>
<tr><td align="left"><code>message_delta</code></td><td align="left">LangChain <code>on_chat_model_stream</code></td><td align="left"><code>{content: str}</code></td><td align="left">拼到对话气泡</td></tr>
<tr><td align="left"><code>tool_call</code></td><td align="left">LangChain <code>on_tool_start</code></td><td align="left"><code>{tool: str, args: dict}</code></td><td align="left">显示"AI 正在使用 X 工具"提示</td></tr>
<tr><td align="left"><code>tool_result</code></td><td align="left">LangChain <code>on_tool_end</code></td><td align="left"><code>{tool: str, output: any}</code></td><td align="left">收起提示 / 展示结构化结果</td></tr>
<tr><td align="left"><code>thinking</code></td><td align="left">路由 yield(可选)</td><td align="left"><code>{text: str}</code></td><td align="left">灰色行显示思考过程</td></tr>
<tr><td align="left"><code>requires_action</code></td><td align="left">LangGraph interrupt</td><td align="left"><code>{action: str, payload: any}</code></td><td align="left">弹按钮"通过 / 拒绝"</td></tr>
<tr><td align="left"><code>run_completed</code></td><td align="left">流自然结束</td><td align="left"><code>{}</code></td><td align="left">关 loader / 允许用户输入</td></tr>
<tr><td align="left"><code>run_failed</code></td><td align="left">try-except 捕获</td><td align="left"><code>{error: str, type: str}</code></td><td align="left">显示错误 toast</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**接 LangChain agent 的关键代码片段**:

In [ ]:
from fastapi import FastAPI, Query
from fastapi.responses import StreamingResponse
from typing import Annotated, AsyncGenerator

app = FastAPI()


def translate(ev: dict) -> tuple[str, dict] | None:
    # 框架事件 → 稳定业务事件(过滤 on_chain_* 等内部噪音)
    name, data = ev.get("name", ""), ev.get("data") or {}
    if ev.get("event") == "on_chat_model_stream":
        text = getattr(data.get("chunk"), "content", "")
        return ("message_delta", {"content": text}) if text else None
    if ev.get("event") == "on_tool_start":
        return ("tool_call", {"tool": name, "args": data.get("input", {})})
    if ev.get("event") == "on_tool_end":
        return ("tool_result", {"tool": name, "output": str(data.get("output"))})
    return None


# 本例走"简单输入档":GET + Query,前端可以直接 new EventSource(url) 订阅(浏览器原生)。
#
# 前端请求示例:
#   GET /chat/events?session_id=s_001&message=帮我查北京今天的天气
#
# 前端代码示意(JS):
#   const es = new EventSource("/chat/events?session_id=s_001&message=...");
#   es.addEventListener("tool_call",     (e) => { /* JSON.parse(e.data) → 渲染工具调用 */ });
#   es.addEventListener("message_delta", (e) => { /* 追加 token 到对话气泡 */ });
#   es.addEventListener("run_completed", (e) => es.close());
#
# 后端返回示例:HTTP/1.1 200 OK + Content-Type: text/event-stream + 业务事件流 ——
#   event: run_started
#   data: {"session_id":"s_001"}
#
#   event: tool_call
#   data: {"tool":"get_weather","args":{"city":"北京"}}
#
#   event: tool_result
#   data: {"tool":"get_weather","output":"晴 18°C"}
#
#   event: message_delta
#   data: {"content":"北京今天"}
#
#   event: message_delta
#   data: {"content":"晴,18 度"}
#
#   event: run_completed
#   data: {}
@app.get("/chat/events")
async def chat_events(
    session_id: Annotated[str, Query(min_length=1)],
    message: Annotated[str, Query(min_length=1, max_length=4000)],
) -> StreamingResponse:
    async def gen() -> AsyncGenerator[str, None]:
        yield sse_pack("run_started", {"session_id": session_id})
        async for ev in agent.astream_events({"messages": [{"role": "user", "content": message}]}, version="v2"):
            if item := translate(ev):
                yield sse_pack(*item)
        yield sse_pack("run_completed", {})

    return StreamingResponse(gen(), media_type="text/event-stream", headers={"X-Accel-Buffering": "no"})


&emsp;&emsp;**5 段式拓展:还有别的方式推多事件吗?**

- **遇到什么问题**:agent 要把"内部多类事件"实时推给前端,前端按类型分发到不同 UI 块。
- **主流方案表**:SSE / WebSocket / HTTP 长轮询 / Server-side push(HTTP/2 push,已被淘汰)。
- **本课怎么选**:SSE。因为 agent 工具调用过程是**服务端单向推送**——客户端不需要反向推数据回去,只需要服务端按节奏推事件。
- **为什么这样选**:SSE 在 HTTP 之上,不需要新协议升级 / 不需要前端额外库 / EventSource 浏览器原生 + 自动断线重连。
- **其他方案什么时候用**:WebSocket 用在双向场景(§12 详述);HTTP 长轮询用在不能跑 SSE 的远古浏览器(几乎不再需要)。

&emsp;&emsp;<font color=red>SSE 还有两条实战层的坑我们要先埋下</font>——

- **EventSource 不能加自定义 header**:SSE 鉴权要走 Query 兜底(`?api_key=...`),普通 POST 仍然可以走 `Authorization: Bearer xxx` header。
- **SSE 长连接经过 nginx 反代时要关 buffer**:`proxy_buffering off` / `proxy_cache off` / `gzip off` 要和 `X-Accel-Buffering: no` 配合。

&emsp;&emsp;**这条接口落到真实项目里**:工具函数(`sse_pack` / `translate`)单独抽一个模块(便于多条 SSE 接口复用);业务事件名锁定一份 enum 给前端共享,典型 7 种—— `run_started` / `message_delta` / `tool_call` / `tool_result` / `requires_action` / `run_completed` / `run_failed`,覆盖一次完整对话从开始到结束 + 工具调用 + 异常的所有 UI 分发场景。

## 8. 文件上传接口

&emsp;&emsp;从这一节开始我们看一组涉及文件的接口——上传(本节)、解析、下载,三者各管一个方向。文件上传接口对外提供的能力很直白:**用户在前端选一份本地文件(CSV / PDF / 图片)传给 agent 处理**。这是 第 1 章 数据分析 agent / 文件处理 agent / RAG 知识库 agent 都要的入口。

&emsp;&emsp;**展现形式**:数据分析 agent 页面上有"上传 CSV 开始分析"按钮,用户点完弹出文件选择窗,选完一份 `sales_2026.csv` → 前端用 `multipart/form-data` 把文件二进制 + 文件名一起发到 `POST /upload` → 后端把文件落盘到 `uploads/` 目录,读成 DataFrame 喂给 agent → 返 `{file_id: 'abc-123', filename: 'sales_2026.csv'}`。或者文件处理 agent 上传 50 页 PDF,前端拿到 `file_id` 后跳到解析进度页继续走。

> <font size=2>**【名词解释】<font color=red>multipart/form-data</font>(多部分表单数据)** — HTTP 协议里专门用来传**文件 + 普通字段混合**的请求体格式。普通 JSON 请求体只能传文本,传不了二进制文件;`multipart/form-data` 把请求体切成多个"part",每个 part 可以是一个字段也可以是一份文件,各自带自己的 Content-Type。浏览器表单的 `<input type="file">` 默认就走这个格式。</font>

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">中等</td><td align="left">主要是文件本身 + 少量字段(<code>file</code> + 可选 <code>description</code>)</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">创建</td><td align="left">把文件落盘到服务端</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">单次上传,完成后返一次</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left"><strong>是,上传方向</strong></td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">上传本身一般不算长任务</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`POST /upload` + `UploadFile` + `multipart/form-data`**。FastAPI 的 `UploadFile` 类是我们用来接收上传文件的对象,可以拿 `.filename` / `.content_type` / `.read()` 等方法操作文件。请求体格式必须是 `multipart/form-data`,不能用 JSON。

> <font size=2>**【名词解释】<font color=red>UploadFile</font>(FastAPI 上传文件类)** — FastAPI 接收上传文件的类型注解。接口签名写 `file: UploadFile` 后,FastAPI 自动从 `multipart/form-data` 请求体里把文件解出来,对象支持流式读 + 保留原文件名 + async API,比读裸 `bytes` 更适合大文件场景。**具体 API 用法 + multipart 编码细节在下面实现段展开**。</font>

### FastAPI 实现: 文件上传接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, File, HTTPException, Request, UploadFile
from pathlib import Path
import uuid

app = FastAPI()
MAX_SIZE = 50 * 1024 * 1024  # 50 MB
UPLOAD_DIR = Path("uploads")


# 前端请求示例:POST /upload   Content-Type: multipart/form-data
#   表单字段:file=<binary 文件内容,比如 report.pdf>
# 后端返回示例:{"file_id": "9f3a1c2b", "filename": "report.pdf", "size": 1048576}
@app.post("/upload")
async def upload(request: Request, file: UploadFile = File(...)) -> dict:
    # 先看 Content-Length,避免明显超大的文件进入读取流程
    if int(request.headers.get("content-length", 0)) > MAX_SIZE:
        raise HTTPException(status_code=413, detail="file too large")

    file_id = uuid.uuid4().hex[:8]
    path, size = UPLOAD_DIR / f"{file_id}_{file.filename}", 0
    with path.open("wb") as f:
        # 分块读取,不要 await file.read() 一次性把大文件塞进内存
        while chunk := await file.read(1024 * 1024):
            size += len(chunk)
            if size > MAX_SIZE:
                path.unlink(missing_ok=True)
                raise HTTPException(status_code=413, detail="file too large")
            f.write(chunk)
    return {"file_id": file_id, "filename": file.filename, "size": size}

&emsp;&emsp;返 `{"file_id":"a3f5b8c1","filename":"test.csv","size":12,"content_type":"text/csv","saved_as":"uploads/a3f5b8c1_test.csv"}`。

&emsp;&emsp;**`UploadFile` 使用要点**:

- **函数签名**:`file: UploadFile = File(...)`,请求体格式必须是 `multipart/form-data`(不能用 JSON,二进制数据要走 multipart 编码)
- **对象常用 API**:`.filename`(原文件名) / `.content_type`(MIME 类型) / `.read()`(读 bytes) / `.write()` / `.seek()` —— 都是 async 方法
- **大文件友好**:内部用流式读,可以分块写入磁盘,不会一次性把整份文件吃进内存(裸 `bytes = File(...)` 会全读进内存,50 MB 文件 = 50 MB 占用)
- **混合上传**:文件 + 普通字段同时传时,普通字段用 `Form(...)` 接(见下方多文件上传示例);**multipart 跟 JSON 不能在同一个请求里混用**——要传 JSON 配置就拆字段走 `Form(...)`

&emsp;&emsp;**`UploadFile` vs 裸 `bytes` 的差别**:FastAPI 还允许 `file: bytes = File(...)` 直接接 bytes,但**整份文件会被一次性读进内存**——50 MB 文件没事,500 MB 就吃内存了。`UploadFile` 内部是 `SpooledTemporaryFile`(小文件内存,大文件落临时盘),还提供 async API,**大文件场景必须用 `UploadFile`**。

&emsp;&emsp;**多文件上传 + 普通字段混合**:

In [ ]:
from fastapi import FastAPI, File, Form, UploadFile

app = FastAPI()


# 前端请求示例:POST /upload-batch   Content-Type: multipart/form-data
#   表单字段:files=<a.pdf>, files=<b.pdf>, description="季度报告打包"
# 后端返回示例:{"description": "季度报告打包", "count": 2, "files": [{"filename": "a.pdf", "size": 12345}, {"filename": "b.pdf", "size": 67890}]}
@app.post("/upload-batch")
async def upload_batch(
    files: list[UploadFile] = File(...),
    description: str = Form(""),  # 普通字段用 Form,不是 Body
) -> dict:
    saved = []
    for f in files:
        contents = await f.read()
        saved.append({"filename": f.filename, "size": len(contents)})
    return {"description": description, "count": len(files), "files": saved}

&emsp;&emsp;<font color=red>multipart 跟 JSON 不能在同一个请求里混用</font>——这是一个高频坑。如果想"传文件 + 传 JSON 配置",必须把 JSON 字段拆开走 `Form(...)`,或者把 JSON 字段当字符串塞进 multipart 的一个 part 里,后端再 `json.loads` 解析。

## 9. 文件解析接口

&emsp;&emsp;接续上一节——用户已经上传了一份文件(拿到 `file_id`),现在我们告诉后端"开始解析"。**这是一个典型的长任务**:OCR(图片转文字)+ 切块 + 抽要点 + 摘要要跑十几分钟,前端不能阻塞等结果,要"提交立刻拿到 `task_id`,然后轮询进度,完事拿结果"。这是文件处理 agent 区别于数据分析 / RAG 的核心接口形态。

&emsp;&emsp;**展现形式**:打开 ChatPaper / Notion AI 文档总结这类产品——用户上传一份 50 页 PDF → 后端立刻返 `{task_id: 'parse-abc', status: 'queued'}` → 前端弹出进度条页面 → **每 2 秒拉一次** `GET /parse/parse-abc/status` 拿到 `{status: 'running', progress: 0.3, current_page: 15, current_step: 'OCR'}` → 进度涨到 100% / status 变 `done` → 前端跳到下载接口拿抽取后的 JSON / 标注 PDF / 摘要报告。

> <font size=2>**【名词解释】<font color=red>BackgroundTasks</font>(FastAPI 后台任务机制)** — FastAPI 提供的"异步任务"工具,我们把一个函数交给它后,FastAPI 立刻返响应给前端,然后**在后台继续跑这个函数**。适合"创建任务时立刻返 task_id,真活在后台跑"这种场景。生产级跨机器异步通常会上 Celery / RQ 等专门的任务队列。**add_task 调用细节 + 多 worker 部署坑在下面实现段展开**。</font>

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">中等</td><td align="left">一个 <code>file_id</code> + 可选解析参数</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">创建</td><td align="left">创建一个解析任务</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">状态查询是请求-响应,不是流式</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left"><strong>是</strong></td><td align="left">上一步上传 + 这一步解析后产物</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left"><strong>是</strong></td><td align="left">这是本节核心维度</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**三段式**——

- **创建解析任务**:`POST /parse` Body `{file_id: 'abc-123'}` → 立刻返 `{task_id: 'parse-abc', status: 'queued'}`,后端用 `BackgroundTasks` 把真活儿放到后台跑
- **轮询进度**:`GET /parse/{task_id}/status` → 返 `{status, progress, current_page, current_step}`
- **取结果**:`GET /parse/{task_id}/result` → 返抽取的结构化 JSON(或跳下载接口)

&emsp;&emsp;这一套和通用长任务三段式本质相同:**文件解析是长任务的一个具体子类**,差别只在进度字段更细,比如多了 `current_page`。看到任何“上传 → 异步处理 → 拿结果”模式,都能套这个三段式。

<div align=center><font size=2 color=#999999>文件解析接口的异步任务时序 — Client 提交 → 立刻返 task_id → 轮询进度 → 完成取结果</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-ade18798.jpg" width=80%></div>
<br>

### FastAPI 实现: 文件解析接口

&emsp;&emsp;**`BackgroundTasks` 使用要点**:

- **函数签名**:声明 `bg: BackgroundTasks` 参数,FastAPI 依赖注入会自动给你一个实例
- **注册任务**:`bg.add_task(func, arg1, arg2, ...)` 把函数 + 参数压进后台执行队列;`add_task` 必须在路由函数 `return` 之前调
- **执行时序**:FastAPI **先把响应发给前端**,然后在**同一个 asyncio 进程**里把 `func(arg1, arg2)` 跑掉——前端拿到 `task_id` 时,后台任务才刚被调度
- **单进程内异步,不是任务队列**:不跨进程、不跨机器。多 worker 部署(`gunicorn -w 4`)下进程间内存不共享,状态字典会"看脸"——这是部署篇要展开解决的坑(下面 5 段式拓展会先给临时方案)

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel
import uuid

app = FastAPI()
_tasks: dict[str, dict] = {}  # 教学用;生产请换 DB / 任务队列


class ParseRequest(BaseModel):
    file_id: str


async def parse_file(parse_id: str, file_id: str) -> None:
    # 后台任务必须自己写状态:running/progress/done/failed
    try:
        _tasks[parse_id].update(status="running", progress=0.0)
        await parser.run(file_id, on_progress=lambda p: _tasks[parse_id].update(progress=p))
        _tasks[parse_id].update(status="done", result={"file_id": file_id})
    except Exception as exc:
        _tasks[parse_id].update(status="failed", error=str(exc))


# 前端请求示例:POST /parse   Body: {"file_id": "9f3a1c2b"}
# 后端返回示例:{"parse_id": "parse-7a8b9c0d", "status": "queued"}   ← 立刻返回,真正解析后台跑
@app.post("/parse")
async def create_parse(req: ParseRequest, bg: BackgroundTasks) -> dict:
    parse_id = f"parse-{uuid.uuid4().hex[:8]}"
    _tasks[parse_id] = {"status": "queued", "progress": 0.0}
    bg.add_task(parse_file, parse_id, req.file_id)  # 立刻返响应,真活儿后台跑
    return {"parse_id": parse_id, "status": "queued"}


# 前端请求示例:GET /parse/parse-7a8b9c0d/status   ← 前端拿到 parse_id 后轮询
# 后端返回示例:{"parse_id": "parse-7a8b9c0d", "status": "running", "progress": 0.6}
#         完成时: {"parse_id": "parse-7a8b9c0d", "status": "done", "progress": 1.0, "result": {"file_id": "9f3a1c2b"}}
@app.get("/parse/{parse_id}/status")
async def parse_status(parse_id: str) -> dict:
    if parse_id not in _tasks:
        raise HTTPException(status_code=404, detail="parse_id not found")
    return {"parse_id": parse_id, **_tasks[parse_id]}

<div align=center><font size=2 color=#999999>文件解析三段式 + BackgroundTasks 时序图 — 创建立刻返 parse_id / 轮询进度 / 完成取结果</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-2e96fd0c.jpg" width=85%></div>
<br>

&emsp;&emsp;**`BackgroundTasks` 的几个细节**:

- **注册顺序很重要**:`bg.add_task(...)` 必须在路由函数 return 之前注册。FastAPI 先把 response 发给前端,然后才执行 `add_task` 注册的所有任务。
- **任务函数可以是同步 / 异步**:同步函数(`def func(...)`)FastAPI 把它放到 starlette 的 threadpool 里执行(默认 40 个线程);异步函数(`async def func(...)`)直接挂到当前 event loop 执行。**坑在哪**:① 同步任务里的 `time.sleep(5)` 会占用一个 threadpool 线程 5 秒——单任务无所谓,**并发 50 个就把池子耗尽**,后续同步路由也会卡住;② 异步任务里如果用了 `time.sleep`(同步版本)会**直接卡住 event loop**——必须用 `await asyncio.sleep`。**通用建议**:I/O bound 长任务用 `async def + await`(模型推理、外部 API);CPU bound 大计算用 `async def + run_in_executor` 把它丢出 event loop;**任务耗时超过几十秒就别用 BackgroundTasks 了,用 Celery / RQ / Arq 等专门队列**。
- **任务异常被吞**:任务函数里抛异常,FastAPI 不会把异常返给前端(响应已经发了);异常会进入服务日志,但没人主动看就漏了。生产环境要用 `try / except` 包住任务函数 + 把错误状态写到任务存储里。

&emsp;&emsp;**5 段式拓展:BackgroundTasks 的适用边界**

- **遇到什么问题**:请求收到后真活儿要跑几秒到几十分钟,前端不能阻塞等结果。
- **主流方案表**:`BackgroundTasks`(本课) / `asyncio.create_task` / Celery / Dramatiq / arq(全部都是 Python 任务队列)。
- **本课怎么选**:`BackgroundTasks`,因为我们的演示场景跑在单机单进程,任务时间短(分钟级)、不需要持久化、不需要重试。
- **为什么这样选**:零额外依赖,FastAPI 原生集成,代码最少。
- **其他方案什么时候用**:跨进程 / 多 worker 共享任务状态 → Celery / arq;任务需要持久化(重启不丢)→ Celery;任务需要重试 / 优先级队列 → Celery / arq。

&emsp;&emsp;<font color=red>注意这个部署坑</font>:上面的 `_tasks` 字典存在进程内,本机单 worker 跑得稳。但生产环境用 gunicorn 起多 worker(`gunicorn -w 4`)时,每个 worker 独立内存空间。task 在 worker A 创建,status 查询请求如果被 nginx 路由到 worker B,就可能直接 404。

&emsp;&emsp;**Interim workaround**(部署篇深入前的临时方案):

<p align="center"><font face="黑体" size=4>Interim workaround:多 worker 状态共享的临时方案</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">方案</th><th align="center">实现</th><th align="center">适用场景</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>共享存储:SQLite/PostgreSQL</strong></td><td align="left">任务状态写表,查询读表(适合需要持久化的场景)</td><td align="left">需要重启不丢任务 / 需要审计日志</td></tr>
<tr><td align="left"><strong>粘性会话(sticky session)</strong></td><td align="left">nginx <code>ip_hash</code>,同一客户端一直路由到同一 worker</td><td align="left">仅短期过渡,新机器或扩容时会失效</td></tr>
<tr><td align="left"><strong>跨进程任务队列</strong></td><td align="left">Celery / arq,状态自带跨进程共享</td><td align="left">任务跑超过几十秒、需要重试 / 优先级</td></tr>
</tbody>
</table>
</div>

## 10. 报告 / 导出下载接口

&emsp;&emsp;走完上传和解析,自然要把产物拿回来——这就是下载接口的事:**后端生成一份文件(JSON / Excel / PDF / 压缩包)让浏览器下载到本地**,而不是把文件内容塞在 JSON 字段里直接返(那样浏览器会显示内容,不会触发下载)。

&emsp;&emsp;**举两个例子来表达具体展现形式**:

1. **待办应用导出全部待办**:用户在右上角点"导出全部待办"按钮 → 前端发 `GET /todos/export` → 浏览器弹出 `todos.json` 下载到本地下载文件夹 → 用户能用任何 JSON 阅读器打开。

2. **数据分析 / 文件处理 agent 下载产物**:跑完一份数据分析 → 前端点"下载分析报告"按钮 → 浏览器弹出 `分析报告.xlsx` 下载;或文件处理 agent 跑完 OCR + 抽取 → 点"下载抽取结果" → 浏览器弹出 `extracted.json`。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">简单</td><td align="left">通常我们用一个资源 ID 或导出参数</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">只读</td><td align="left">—</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left"><strong>是,下载方向</strong></td><td align="left">—</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">假设报告已生成好;若需要现场生成大报告,我们先走长任务接口再下载</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`GET + FileResponse`**——带 `Content-Disposition: attachment; filename=xxx` 响应头让浏览器走下载流程而不是预览。`FileResponse` 是 FastAPI 的一个 Response 子类,直接把磁盘上的文件流给前端。

> <font size=2>**【名词解释】<font color=red>FileResponse</font>(FastAPI 文件响应类)** — FastAPI 用来"把磁盘文件直接返给前端"的 Response 子类。自动按文件扩展名设 `Content-Type`,可以自定义 `filename` 触发浏览器下载弹窗,内部用流式读避免大文件内存爆。**具体参数 + 跟 `JSONResponse` 的浏览器行为差异在下面实现段展开**。</font>

### FastAPI 实现: 报告 / 导出下载接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI
from fastapi.responses import FileResponse

app = FastAPI()


# 前端请求示例:GET /reports/rpt_2026Q1   ← 浏览器直接点链接 or <a download>
# 后端返回示例:Content-Type: application/json
#         Content-Disposition: attachment; filename="rpt_2026Q1.json"
#         <文件二进制内容>   ← 浏览器自动弹下载
@app.get("/reports/{report_id}")
async def download_report(report_id: str) -> FileResponse:
    # 关键:先把导出内容写成一个文件,再用 FileResponse 交给浏览器下载
    path = export_service.write_report(report_id)
    return FileResponse(
        path=path,
        filename=path.name,                 # 触发下载,值即下载文件名
        media_type="application/json",      # 真实项目按格式换:json / xlsx / pdf / zip
    )

&emsp;&emsp;**`FileResponse` 使用要点**:

- **基本调用**:`FileResponse(path, filename=..., media_type=...)`——`path` 是磁盘路径,后两个参数可选
- **MIME 类型**:不传 `media_type` 时自动按文件扩展名推断(`.json` → `application/json` / `.xlsx` → `application/vnd.openxmlformats-officedocument.spreadsheetml.sheet`);需要强制覆盖时显式传 `media_type=...`
- **触发下载弹窗**:传 `filename=...` 会在响应头加 `Content-Disposition: attachment; filename=...`,浏览器走下载流程而不是新标签页预览(这是 `FileResponse` 跟普通 `JSONResponse` 在浏览器侧的关键差别)
- **大文件友好**:内部按块流式读,百 MB 级文件不会一次性吃内存——这点跟 `UploadFile` 的流式语义对称

&emsp;&emsp;**`FileResponse` 跟普通 `JSONResponse` 的关键差别**:同样是返 JSON,如果用 `JSONResponse` 返,浏览器会**在新标签页显示文本内容**;用 `FileResponse(filename=...)` 返,浏览器会**弹下载窗存盘**。差别就在 `Content-Disposition: attachment` 这个响应头——它告诉浏览器"这是下载,不是预览"。

&emsp;**真实项目里的两处常见扩展**:① **时间戳格式化** —— 导出文件里的 epoch 时间戳要转成人类可读 ISO 字符串(`2026-05-11T16:30:15+08:00` 而不是 `1715000000.0`),不然用户打开 JSON 看到一串数字一头雾水;② **路径管理** —— 导出文件存哪儿不要写死路径,从 FastAPI 的 `lifespan` 注入(`request.app.state.export_dir`),既跨平台又方便测试时换目录。

## 11. 长任务接口

&emsp;&emsp;**文件解析是长任务的一个具体子类——它特化在"输入是文件、解析步骤可枚举(OCR/切块/抽取)"**。本节我们把长任务的**通用形态**抽出来讲清——agent 跑一件几分钟到几十分钟才能完的事,前端不能阻塞等响应,要"提交后立刻收到 `task_id` → 然后定期查进度 → 最后拿结果"。代码 agent 跑改造任务、工作流 agent 跑一条多节点流程、数据分析 agent 跑大表分析都属于这一通用模式;文件解析就是把它特化到"文件场景"的版本。**两节看完后,看到任何"提交 → 异步处理 → 拿结果"模式都能直接套同一套三段式**。

&emsp;&emsp;**展现形式**:打开 Devin / Manus 这类代码 agent,告诉它"帮我把这个 JS 项目改造成 TypeScript" → 前端发 `POST /tasks` Body 包含任务描述 → 后端**立刻**返 `{task_id: 'abc-123', status: 'queued'}` 不等任务跑完 → 前端开始**每 3 秒发** `GET /tasks/abc-123/status` 拿到 `{status: 'running', progress: 0.3, current_step: '分析 src/utils.js'}` → 进度涨到 100% / status 变 `done` → 前端发 `GET /tasks/abc-123/result` 拿到改造后的代码 diff。

&emsp;&emsp;**为什么不用同步聊天接口?**如果我们用同步接口跑一个 10 分钟的任务,HTTP 请求会**挂在那里 10 分钟等响应**——浏览器 / 反向代理 / 客户端通常都有 30-60 秒超时,挂这么久肯定 504 网关超时;就算没超时,前端这 10 分钟里转圈用户体验也很差。**长任务必须把"创建"和"取结果"拆成两次请求,中间靠 task_id 维系**——这是任务执行 agent 接口设计的根本形态。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">中等到复杂</td><td align="left">任务描述 + 配置字段</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">创建</td><td align="left">创建一个任务</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left">否</td><td align="left">状态查询本身是请求-响应,不是流式</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">否</td><td align="left">—</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">看场景</td><td align="left">代码改造任务最后产物是 diff 文件,文件处理任务参考文件解析那节;通用长任务可能就返 JSON</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left"><strong>是</strong></td><td align="left">这是本节核心维度</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**三段式 + 两条辅助**——

- `POST /tasks` Body `{...}` → 立刻返 `{task_id, status: 'queued'}`,真活儿丢给 `BackgroundTasks` 后台跑
- `GET /tasks/{task_id}/status` → 返 `{status, progress, current_step}`,前端按节奏轮询(**轮询节奏典型 1-3 秒用于交互式前台任务、5-10 秒用于后台长任务**;对实时性要求高也可用 SSE 推送 `tasks_status` 事件替代轮询)
- `GET /tasks/{task_id}/result` → status=done 时取最终产物
- **`DELETE /tasks/{task_id}` 取消接口**(常用辅助):用户中途反悔,前端发删除请求,后端把任务标记为 `cancelled` + 停掉相关后台进程。生产级长任务接口通常都要有。
- **`POST /tasks/{task_id}/retry` 重试接口**(可选):任务失败(`status=failed`)后,前端发重试请求重新跑一次。简单场景可以省,但模型推理 / 数据分析这类任务因网络抖动失败的概率不低,有重试体验会好很多。

> <font size=2>**【架构提示】<font color=red>task_id 从哪儿来、状态存哪儿</font>** — `task_id` 是后端生成的唯一标识符,通常是 UUID 或带前缀的短 ID。**任务的状态 + 进度 + 结果存在哪里**决定了这套接口能不能在多 worker 部署下可靠工作。最简实现是内存 dict,本机单 worker 跑得通;多 worker 会因为状态不共享而失效。更稳的实现是 SQLite / PostgreSQL 任务表。</font>

&emsp;&emsp;**和文件解析的关系**:**文件解析是本节通用三段式在"文件场景"的特化版本**——`task_id` 在文件解析里通常叫 `parse_id`,`current_step` 在文件解析里更具体(`current_page` / 当前正在跑 OCR 还是切块),其他完全相同。我们把这两节学完后,看到任何"提交 → 异步处理 → 拿结果"模式都能直接套这套三段式 + 取消 / 重试的接口骨架。

<div align=center><font size=2 color=#999999>长任务接口三段式时序 — 提交立刻返 task_id → 轮询查进度 → 完成取结果</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-6993cca5.jpg" width=80%></div>
<br>

### FastAPI 实现: 长任务三段式接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel, Field
import uuid

app = FastAPI()
_tasks: dict[str, dict] = {}  # 教学用;生产请换 DB / 任务队列


class TaskCreate(BaseModel):
    description: str = Field(..., min_length=1, max_length=2000)


# 前端请求示例:POST /tasks   Body: {"description": "分析过去 30 天销售数据并出报告"}
# 后端返回示例:{"task_id": "task-7a8b9c0d", "status": "queued"}   ← 立刻返,真活儿后台跑
@app.post("/tasks")
async def create_task(req: TaskCreate, bg: BackgroundTasks) -> dict:
    task_id = f"task-{uuid.uuid4().hex[:8]}"
    _tasks[task_id] = {"status": "queued", "progress": 0.0}
    bg.add_task(worker.run, task_id, req.description)
    return {"task_id": task_id, "status": "queued"}


# 前端请求示例:GET /tasks/task-7a8b9c0d/status   ← 前端轮询拿进度
# 后端返回示例:{"task_id": "task-7a8b9c0d", "status": "running", "progress": 0.4, "current_step": "清洗缺失值"}
@app.get("/tasks/{task_id}/status")
async def task_status(task_id: str) -> dict:
    if task_id not in _tasks:
        raise HTTPException(status_code=404, detail="task not found")
    return {"task_id": task_id, **_tasks[task_id]}


# 前端请求示例:DELETE /tasks/task-7a8b9c0d   ← 用户点"取消"
# 后端返回示例:{"task_id": "task-7a8b9c0d", "status": "cancelled"}
@app.delete("/tasks/{task_id}")
async def cancel_task(task_id: str) -> dict:
    # 生产还要通知 worker 真正停下,这里只改状态
    _tasks[task_id]["status"] = "cancelled"
    return {"task_id": task_id, "status": "cancelled"}

&emsp;&emsp;<b>为什么单独开一节?</b>因为文件解析那节的画面是"上传 → 解析"很具体,容易让人以为这套机制只对文件场景适用。<b>抽出来单独讲一节,是让我们看到"提交 → 异步处理 → 拿结果"的通用形态</b>——代码改造任务、数据分析任务、工作流执行任务都套同一套三段式 + `BackgroundTasks`。

&emsp;&emsp;**取消 / 重试接口**:

- `DELETE /tasks/{task_id}` 取消——用户中途反悔,前端发删除请求,后端把状态标 `cancelled` + 通知 worker 停。生产级长任务接口通常都要有,因为大模型推理任务跑十几分钟然后用户改主意是常态。
- `POST /tasks/{task_id}/retry` 重试——任务失败(`status=failed`)后,前端发重试请求,后端用同样的输入重新跑一次。可选,简单场景可以省。

&emsp;&emsp;**轮询节奏**:前端轮询节奏典型 1-3 秒一次(交互式前台任务,用户在等)/ 5-10 秒一次(后台长任务,用户切走了)。如果对实时性要求很高(用户全程盯着进度条),可以用 SSE 的方式推 `tasks_status` 事件替代轮询——但这只是优化选项,**轮询是基础方案,SSE 是优化方案**。

## 12. 双向实时接口

&emsp;&emsp;最后一类业务接口是**双向实时**——客户端和服务端任一方都能随时发新数据,而且通常涉及二进制流双向高频传输。这一类我们只看一种 agent 形态:**实时语音 agent**。

&emsp;&emsp;**展现形式**:实时语音 agent 的典型产品是 ChatGPT 高级语音模式 / OpenAI Realtime API / 各种"打电话给 AI"工具——**用户一边说话,前端实时把麦克风音频流推给后端;后端 ASR 转文字 + agent 推理 + TTS 合成语音,一边生成一边把语音流推回前端播放;用户中途想打断,前端再推一段新的语音过去,后端立刻停止当前播放切到新输入**。整个过程音频流双向同时跑,毫秒级延迟。

> <font size=2>**【名词解释】<font color=red>WebSocket</font>(双向二进制长连接协议)** — 基于 HTTP 升级握手建立的全双工长连接协议。一旦建立,客户端和服务端可以**任意时间**互相发数据(不再受请求-响应模式约束),且支持二进制帧 + 低延迟。浏览器有原生 `WebSocket` API,FastAPI 支持 `@app.websocket("/ws")` + `async def ws(websocket: WebSocket)` 接收双向消息。</font>

&emsp;&emsp;**为什么我们用过的 HTTP / SSE 都不够?**

- **SSE 是单向**——服务端推、客户端不能反推音频流。语音 agent 的"用户中途说话打断"做不了——SSE 上客户端根本没有"反向推音频"的能力。
- **音频是二进制流 + 高频小包**——HTTP 请求-响应模型每次都要建连接,延迟和开销受不了。
- **WebSocket 长连接 + 双向二进制 + 低延迟** → 实时语音的唯一合理选择。

<p align="center"><font face="黑体" size=4>6 维度过一遍</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">维度</th><th align="center">取值</th><th align="center">说明</th></tr>
</thead>
<tbody>
<tr><td align="left">维度 1(输入复杂度)</td><td align="left">特殊</td><td align="left">不是一次性传整段数据,而是持续传二进制音频帧</td></tr>
<tr><td align="left">维度 2(改状态)</td><td align="left">特殊</td><td align="left">长连接,整个会话期间持续读写</td></tr>
<tr><td align="left">维度 3(实时)</td><td align="left"><strong>是,而且双向</strong></td><td align="left">超过维度 3 通常理解的"单向流式",到了"双向流式"</td></tr>
<tr><td align="left">维度 4(结构化事件)</td><td align="left">看场景</td><td align="left">纯音频帧不需要,但实际产品里通常会混控制事件(<code>speech_start</code> / <code>speech_end</code> / <code>interrupt</code>)</td></tr>
<tr><td align="left">维度 5(涉及文件)</td><td align="left">否</td><td align="left">音频流不算文件</td></tr>
<tr><td align="left">维度 6(长任务)</td><td align="left">否</td><td align="left">每条用户对话本身不长,但会话长连接持续</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**HTTP 形态**:**`WebSocket`**——协议层升级,不再是普通 HTTP。FastAPI 用 `@app.websocket("/voice")` 装饰器声明,接口函数签名是 `async def voice(ws: WebSocket)`,函数体内 `await ws.accept()` 然后用 `await ws.receive_bytes()` / `await ws.send_bytes()` 双向读写音频帧。

&emsp;&emsp;**WebSocket 真用得上的其他场景**:多人协作编辑(Figma / Notion 多人光标)/ 实时白板 / 实时游戏 / 在线 IDE 协作。这些场景的共同特征是"多端任一时刻都可能发新数据 + 延迟敏感"——和实时语音同源。

&emsp;&emsp;**SSE 单向 vs WebSocket 双向**对比:我们把两者的区别用一张图立清。

<div align=center><font size=2 color=#999999>SSE 单向 vs WebSocket 双向:协议形态对比</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-87f80037.jpg" width=80%></div>
<br>

&emsp;&emsp;<font color=red>选 SSE 还是 WebSocket 我们就看一件事:客户端要不要"在不发新 HTTP 请求的前提下"主动推数据给服务端</font>。要 → WebSocket;不要(只是服务端推给客户端)→ SSE。SSE 简单、能复用 HTTP 基础设施;WebSocket 复杂、要专门处理长连接维护和断线重连,但能做双向。

### FastAPI 实现: 实时语音双向接口

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
import logging

app = FastAPI()
logger = logging.getLogger("ws-demo")


# 前端请求示例:WebSocket 握手 —— ws://server/voice   (协议升级,不是 HTTP)
#         握手成功后双向传二进制帧:前端推麦克风音频帧 → 后端收到 → 这里 echo 回去
# 后端返回示例:握手响应 101 Switching Protocols + 之后持续推音频帧
#         (本 demo 是 echo:前端发 1024 字节,后端就把同样 1024 字节回过去)
@app.websocket("/voice")
async def voice(ws: WebSocket) -> None:
    """实时语音双向接口 echo demo — 收到啥就发回去啥。"""
    await ws.accept()
    logger.info("WebSocket connected")
    try:
        while True:
            # receive_bytes 收二进制(音频帧);receive_text 收文本(控制消息)
            data = await ws.receive_bytes()
            logger.info(f"received {len(data)} bytes, echoing back")
            await ws.send_bytes(data)
    except WebSocketDisconnect:
        logger.info("WebSocket disconnected")

<p align="center"><font face="黑体" size=4>SSE vs WebSocket 怎么选</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">维度</th><th align="center">SSE</th><th align="center">WebSocket</th></tr>
</thead>
<tbody>
<tr><td align="left">方向</td><td align="left">服务端 → 客户端(单向)</td><td align="left">双向</td></tr>
<tr><td align="left">数据类型</td><td align="left">文本(可序列化 JSON)</td><td align="left">文本 + 二进制</td></tr>
<tr><td align="left">协议层</td><td align="left">HTTP(普通 GET)</td><td align="left">HTTP Upgrade → ws/wss</td></tr>
<tr><td align="left">客户端 API</td><td align="left"><code>EventSource</code>(原生 + 自动重连)</td><td align="left"><code>WebSocket</code>(原生,断线需自己处理)</td></tr>
<tr><td align="left">鉴权</td><td align="left">受 EventSource 限制(只能 Query 或 cookie)</td><td align="left">握手时 Header 可用</td></tr>
<tr><td align="left">反代复杂度</td><td align="left">nginx 需关 buffer</td><td align="left">nginx 需配 Upgrade header</td></tr>
<tr><td align="left"><strong>本课用在</strong></td><td align="left">工具调用过程接口 / 日志流 / 任务进度推送</td><td align="left">实时语音 / 多人协作 / 实时游戏</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**判断标准**:客户端要不要"在不发新 HTTP 请求的前提下"主动推数据给服务端?要 → WebSocket;不要(只是服务端推给客户端)→ SSE。绝大多数 agent 接口是单向推(进度 / 工具事件 / token 流),用 SSE 就够;只有实时语音 / 多人协作这种"双向同时通信"的场景才用 WebSocket。

&emsp;&emsp;**多数 agent 用不到 WebSocket**——只要是"对话 + 工具调用 + 文本流"这一类,普通流式 + SSE 已经够用。这一节立一个判断意识:**看到一条业务接口需要"双向 + 实时 + 二进制",就该想 WebSocket;否则 SSE 优先**。实时语音 / 多人协作编辑 / 实时游戏这几类才真用得上 WebSocket;一般的 chat agent / 工具调用 agent / 数据分析 agent 全部留在 SSE 范畴内即可。

## 13. HTTP 形态总表:业务出口 × 协议形态

<p align="center"><font face="黑体" size=4>第二章 11 类业务接口的 HTTP 形态总表</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">业务接口</th><th align="center">来自哪类 agent</th><th align="center">核心 6 维度命中</th><th align="center">HTTP 形态</th><th align="center">schema 层可扩展点</th></tr>
</thead>
<tbody>
<tr><td align="left">会话历史 / 任务状态接口</td><td align="left">对话助手 / 任务执行 / 工作流 / 数据分析 等</td><td align="left">只读 + 简单输入</td><td align="left"><code>GET + Path / Query</code></td><td align="left">—</td></tr>
<tr><td align="left">待办 / 文档管理接口(CRUD)</td><td align="left">对话助手会话管理 / 工作流 CRUD / RAG 文档 CRUD / 文件处理库</td><td align="left">创建 / 改 / 删 / 简单输入</td><td align="left"><code>GET / POST / PATCH / PUT / DELETE</code> + 资源式路由</td><td align="left">同一套 CRUD 套路可直接承载:<strong>多 agent 角色配置</strong>(`/agents/{id}`)/ <strong>工作流定义</strong>(`/workflows/{id}`)</td></tr>
<tr><td align="left">聊天接口</td><td align="left">对话助手 / 几乎所有带对话能力的 agent</td><td align="left">输入复杂 + 创建</td><td align="left"><code>POST + Body + Pydantic</code></td><td align="left">响应体可加 <strong>citations 字段</strong>:`reply: str` + `citations: list[{doc_id, page, snippet}]` → RAG 类回复"带引用来源"</td></tr>
<tr><td align="left">聊天流式接口</td><td align="left">对话助手 / 工具调用 / RAG / 文件处理 等</td><td align="left">输入复杂 + 创建 + 实时 + 纯文本</td><td align="left"><code>POST + StreamingResponse(text/plain)</code></td><td align="left">—(纯文本流;复杂 schema 走下面的 SSE)</td></tr>
<tr><td align="left">工具结果接口</td><td align="left">工具调用 / RAG / 文件处理</td><td align="left">只读 + 结构化数据</td><td align="left"><code>GET + JSONResponse</code></td><td align="left">可演化成 <strong>RAG 检索结果预览接口</strong>(`GET /search?q=...` → top-K + 分数 + 高亮)</td></tr>
<tr><td align="left">工具调用过程接口 / 日志流接口</td><td align="left">工具调用 / 数据分析 / 代码 agent / 多智能体</td><td align="left">实时 + <strong>结构化事件</strong></td><td align="left"><code>GET + StreamingResponse(text/event-stream)</code>(SSE)+ 业务事件协议</td><td align="left">SSE 事件 schema 可继续扩:<strong>图表中间产物</strong>(`event: chart`,data 内联 base64 / URL)/ <strong>代码 diff 事件</strong>(`event: diff`,data 带 file + patch)/ <strong>跨 agent 消息流</strong>(data 带 `from_agent` / `to_agent` / `round`)/ <strong>半双工确认请求</strong>(`event: need_confirm`,前端再发 `POST /confirm` 回去)</td></tr>
<tr><td align="left">文件上传接口</td><td align="left">数据分析 / RAG / 文件处理</td><td align="left">涉及文件(上传)</td><td align="left"><code>POST + UploadFile + multipart/form-data</code></td><td align="left">—</td></tr>
<tr><td align="left">文件解析接口</td><td align="left">文件处理</td><td align="left">涉及文件 + 长任务</td><td align="left"><code>POST + BackgroundTasks</code> + 三段式(创建 + 状态 + 结果)</td><td align="left">—(本质是下面长任务三段式的"涉及文件"特化)</td></tr>
<tr><td align="left">报告 / 导出下载接口</td><td align="left">对话助手 / 数据分析 / 文件处理</td><td align="left">涉及文件(下载)</td><td align="left"><code>GET + FileResponse</code></td><td align="left">—</td></tr>
<tr><td align="left">长任务接口(三段式)</td><td align="left">任务执行 / 工作流 / 数据分析 / 代码 agent</td><td align="left">长任务 + 状态生命周期</td><td align="left"><code>POST /tasks</code> + <code>GET status</code> + <code>GET result</code> + 可选取消 / 重试</td><td align="left">同一套三段式可承载:<strong>代码改造任务</strong> / <strong>工作流执行</strong> / <strong>大表分析</strong>;status 可带 `current_step` 反映节点进度</td></tr>
<tr><td align="left">双向实时接口</td><td align="left">实时语音 / 多人协作 / 实时控制类</td><td align="left">实时 + 双向 + 二进制</td><td align="left"><code>WebSocket</code></td><td align="left">—(本课只涉及语音双向)</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;我们横着扫一遍,有几条规律值得记住:

- **普通请求-响应是基础盘**——大多数 agent 接口仍然是 `GET/POST/PATCH/DELETE` + JSON / 文件响应,FastAPI 处理这些就是日常工作。
- **聊天能力分两层**——非流式聊天先讲清 `POST + Body + Pydantic`,流式聊天只是在返回方式上升级成 `StreamingResponse(text/plain)`。
- **工具能力也分两层**——工具结果是事后取 JSON,工具过程是实时推 SSE 事件。先结果、后过程,理解成本最低。
- **文件能力形成闭环**——上传→ 解析→ 下载,这三节连在一起看就是文件类 agent 的主流程。
- **长任务三段式是通用抽象**——文件解析只是它的文件版;代码改造、工作流执行、大表分析也能套同一套 task_id 生命周期。
- **WebSocket 只放在双向实时**——绝大多数 agent 用不到,只有实时语音、多人协作、实时控制这类场景才需要。

## 14. 完整接口必备的辅助组件

### 14.1 鉴权机制的实现

&emsp;&emsp;任何一条接口上线后,都得先校 API Key 才让进——不校验,公网上任何人都能拿到 endpoint 调你的 LLM、改你的待办、读你的会话历史。鉴权是拓展层,所有接口都要带。这一节我们详讲 `Depends` 依赖注入 + Header / Query 双来源 + `HTTPException(401)` 的完整实现。

&emsp;&emsp;**先看反例,没有 `Depends` 怎么不行**:

```python
    # 反例:每个接口都重复一段鉴权逻辑
    @app.post("/chat")
    async def chat(req: ChatRequest, request: Request):
        api_key = request.headers.get("x-api-key")
        if not api_key or api_key != "secret-key-123":
            raise HTTPException(status_code=401, detail="unauthorized")
        # ...业务代码...

    @app.get("/sessions/{session_id}/messages")
    async def get_messages(session_id: str, request: Request):
        api_key = request.headers.get("x-api-key")
        if not api_key or api_key != "secret-key-123":
            raise HTTPException(status_code=401, detail="unauthorized")
        # ...业务代码...

    # ...十几个接口,每个都重复这 4 行...
```

&emsp;&emsp;**问题**:① 每个接口都重复——改一处要改十几处;② 鉴权逻辑跟业务逻辑混在一起,可读性差;③ 想给所有接口统一加鉴权时没法批量。我们需要的是**把鉴权抽成一个可复用的依赖**,然后路由声明 `Depends(verify_api_key)` 自动套上——这是 FastAPI 的 `Depends` 依赖注入机制干的事。

> <font size=2>**【名词解释】<font color=red>Depends</font>(FastAPI 依赖注入)** — FastAPI 把"依赖项"抽成可复用的 callable(函数或类),路由签名声明 `dep: ReturnType = Depends(provider_func)`,FastAPI 在执行路由前先跑 `provider_func()`,把返回值注入到 `dep` 参数。依赖函数本身也可以用 `Depends`(嵌套依赖),也可以抛 `HTTPException`(中断请求)。常用场景:鉴权 / 数据库连接 / 配置注入 / 当前用户解析。</font>

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, Depends, Header, HTTPException, Query, status
from typing import Annotated

API_KEY = "..."  # 从配置 / 环境变量读,不写死

# ─────────────────────────────────────────────────────────────────────
# 把"取 key"这一步抽成纯函数(易测、跟 FastAPI 解耦)。
# 两个来源同时支持:
#   - Header:`Authorization: Bearer xxx`  —— 普通 HTTP 客户端推荐(curl / fetch / requests)
#   - Query: `?api_key=xxx`                —— 给那些"加不了自定义 header"的客户端兜底:
#                                              · EventSource(SSE 规范限制只能 GET + 不支持 header)
#                                              · <a href="..."> 文件下载(浏览器直接跳转,没法塞 header)
# ─────────────────────────────────────────────────────────────────────
def _extract(authorization: str | None, api_key: str | None) -> str | None:
    # 优先解 Bearer Token —— 行业惯例 `Authorization: Bearer <token>` 是主流
    if authorization and authorization.lower().startswith("bearer "):
        return authorization.split(maxsplit=1)[1]
    # 退路:① Authorization 不是 Bearer 格式就当裸 token 用  ② 没 header 就退到 Query
    return authorization or api_key


# ─────────────────────────────────────────────────────────────────────
# FastAPI 依赖项 —— 路由签名挂上 `Depends(verify_api_key)` 后,每次请求自动跑这一段。
# 参数声明 `Annotated[str | None, Header()] = None`:
#   - `Header()` 告诉 FastAPI "从请求头里取 Authorization 字段"
#   - `Query()`  告诉 FastAPI "从 URL Query 里取 api_key 字段"
#   - `| None = None`:两个都是可选,缺哪个就传 None
# 请求示例 → 后端行为:
#   ① 普通 HTTP:GET /todos  Header: Authorization: Bearer secret123
#        → authorization="Bearer secret123", api_key=None → _extract 返 "secret123" → 通过
#   ② SSE 兜底:GET /chat/events?api_key=secret123&...
#        → authorization=None, api_key="secret123" → _extract 返 "secret123" → 通过
#   ③ 两个都没传 / 错的:
#        → _extract 返 None / 错值 → 抛 HTTPException(401)
#          FastAPI 自动序列化成:HTTP/1.1 401 Unauthorized + WWW-Authenticate: Bearer + {"detail":"invalid api key"}
# ─────────────────────────────────────────────────────────────────────
async def verify_api_key(
    authorization: Annotated[str | None, Header()] = None,
    api_key: Annotated[str | None, Query()] = None,
) -> None:
    if _extract(authorization, api_key) != API_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="invalid api key",
            # WWW-Authenticate 响应头是 HTTP 401 的标准搭档,告诉客户端"应该用哪种鉴权方案重试"。
            # 浏览器看到这个头会弹出 Basic Auth 输入框等 —— 这里我们指明用 Bearer scheme。
            headers={"WWW-Authenticate": "Bearer"},
        )
    # 注意函数返 None —— 依赖项不需要返值,鉴权失败靠抛异常表达;
    # 真要从 token 反查用户身份(谁调的),把 User 对象 return 出去给路由 Depends 接住即可。


# ─────────────────────────────────────────────────────────────────────
# 挂载粒度:这里在 app 全局挂 —— 所有路由都会先过 verify_api_key 这道关。
# 另两种粒度(路由组 / 单路由)看下面对比表。
# ─────────────────────────────────────────────────────────────────────
app = FastAPI(dependencies=[Depends(verify_api_key)])


&emsp;&emsp;**三种鉴权挂法对比**(单路由 / 路由组 / 全局,生产里按粒度选):

<p align="center"><font face="黑体" size=4>三种鉴权挂法对比</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">接口需求</th><th align="center">FastAPI 知识</th></tr>
</thead>
<tbody>
<tr><td align="left">复用鉴权逻辑</td><td align="left"><code>Depends(...)</code> 依赖注入</td></tr>
<tr><td align="left">读 Header</td><td align="left"><code>Header(alias=...)</code></td></tr>
<tr><td align="left">读 Query</td><td align="left"><code>Query(alias=...)</code></td></tr>
<tr><td align="left">鉴权失败</td><td align="left"><code>HTTPException(401)</code></td></tr>
<tr><td align="left">路由级统一鉴权</td><td align="left"><code>@app.post(..., dependencies=[Depends(...)])</code></td></tr>
<tr><td align="left">路由组级统一鉴权</td><td align="left"><code>APIRouter(dependencies=[Depends(...)])</code></td></tr>
<tr><td align="left">全局级统一鉴权</td><td align="left"><code>app.include_router(router, dependencies=[...])</code></td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**为什么同时支持 Header 和 Query 两种 key 来源?** EventSource 不能加自定义 header,SSE 鉴权要走 Query;普通接口走 `Authorization: Bearer xxx` Header 更干净。文件下载用 `<a href>` 触发时也常走 Query。一份鉴权函数支持两种来源,所有接口共用。

&emsp;&emsp;**5 段式拓展:鉴权方案有几种?**

- **遇到什么问题**:接口要校"谁能调",防止公网随意调用。
- **主流方案表**:API Key(本课)/ Bearer Token / OAuth2 / JWT / Session + Cookie。
- **本课怎么选**:API Key in Header(`Authorization: Bearer <key>`)+ Query 兜底。因为我们的演示场景是"单用户 + 单 key"——你自己跑这个服务、给自己 / 朋友 / 前端用,一个 key 就够,不需要多用户管理。
- **为什么这样选**:简单(一行 `if key == API_KEY`)+ 跟 OpenAI / Claude / OpenRouter 等 agent 平台事实标准对齐(都是 `Authorization: Bearer`)+ EventSource 兼容(走 Query 兜底)。
- **其他方案什么时候用**:多用户场景 → JWT(stateless,不需要 session 表);第三方应用接入 → OAuth2(用户授权流程);Web 应用配合浏览器 → Session + Cookie。

&emsp;&emsp;**这套实现一次覆盖 4 种典型客户端场景**:① `/chat`(POST,普通 JSON 客户端走 `Authorization: Bearer`) / ② `/chat/stream`(POST 流式,同上) / ③ `/chat/events`(GET + EventSource,**走 `?api_key=...` 兜底**,因为 EventSource 不能加自定义 header) / ④ `/reports/{id}`(GET + `<a href>` 直接下载,**也走 Query 兜底**)。生产里再加路由分组级覆盖——`app.include_router(some_router, dependencies=[Depends(verify_api_key)])`,某些公开 endpoint(如 `/health`)不挂依赖即可。

### 14.2 跨域支持的实现

&emsp;&emsp;浏览器有同源策略——前端 `http://localhost:12233`(运行前端的端口)直接 `fetch('http://localhost:12234/chat')`(我们后端的端口),会被浏览器拦下来报 CORS 错误,根本到不了后端。前置课的案例里已经详讲过 `CORSMiddleware` 三件套(`allow_origins` / `allow_methods` / `allow_headers`),这一节我们快速激活前置课内容,然后补一个 agent 场景的特殊点——**EventSource 跨域的额外限制**。

&emsp;&emsp;**FastAPI 表现形式骨架**(直接照搬前置课模板):

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:12233", "https://my-frontend.com"],
    # 开发期偷懒用 ["*"],生产**必须**列具体域名 + allow_credentials=False
    # 浏览器规范禁止 * + credentials 组合,所以两者只能选其一
    allow_cederntials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["X-Request-ID"],  # 前端想从响应里读到的自定义 header 要 expose
)

> <font size=2>**【名词解释】<font color=red>CORSMiddleware</font>(FastAPI 跨域中间件)** — FastAPI 内置的 CORS(Cross-Origin Resource Sharing,跨域资源共享)中间件,在响应里自动加 `Access-Control-Allow-Origin` / `Access-Control-Allow-Methods` 等响应头,以及处理 OPTIONS 预检请求。配置项 `allow_origins` 列允许的前端来源,`allow_methods` / `allow_headers` 控制允许的方法和请求头。</font>

&emsp;&emsp;**agent 场景的特殊点:EventSource 跨域**。前置课讲过普通 `fetch` 跨域的完整规则——但 SSE 走的是 `EventSource`,它的跨域规则**更严**:

- **不能加自定义请求头**——所以 EventSource 鉴权要走 `?api_key=...` Query。
- **不能用 `credentials: 'include'` 跨域带 cookie**——必须把 `withCredentials = true` 显式设上,而且服务端 CORS 配置要 `allow_credentials=True` + 列出明确 origin。

&emsp;&emsp;**真实项目里跨域几乎是默认状态**——前端跑在 `http://localhost:12233`、后端跑在 `http://localhost:12234`,端口不同就是跨域,**必须正确配 CORS**,不然浏览器直接把请求拦下来,后端连日志都看不到。开发期 `allow_origins=["*"]` 简单,生产必须列具体域名(同时 `allow_credentials=False`,浏览器禁止 `*` + credentials 组合)。

&emsp;&emsp;<font color=red>CORS 还要注意网关层一致性</font>:如果 nginx 端和 FastAPI 端返回的 `Access-Control-Allow-Origin` 不一致,浏览器同样会拒绝。

### 14.3 配置管理的实现

&emsp;&emsp;前面所有节代码里我们都把配置硬编码——`API_KEY = "secret-key-123"`、模型名、端口号、CORS 来源……上线前必须把这些抽出来,统一从环境变量 / `.env` 文件加载。这一节我们把这层"配置加载模块"建好,后续所有 router 都从这里 import 用。

In [ ]:
# config.py — 统一配置加载层,全项目就这一份
import os
from dotenv import load_dotenv  # python-dotenv 库,pip install python-dotenv

# 启动时把 .env 文件里的字段灌进 os.environ
# Docker / k8s 部署时不一定有 .env 文件,load_dotenv() 找不到文件不会报错——
# 容器场景直接通过环境变量传配置,跟本地 .env 走的是同一套 os.environ。
load_dotenv()


# ─── 必填配置:用 os.environ[KEY],缺字段直接 KeyError 启动挂 ───
# 这是 "启动时挂 vs 运行时挂" 的关键——缺字段时部署脚本 / CI 能立刻拦截,
# 而不是等业务调到 LLM 那一行才 500。别用 os.getenv("KEY"),默认返 None。
OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
API_KEY = os.environ["API_KEY"]


# ─── 可选配置:os.getenv(KEY, "默认值") ───
LLM_MODEL = os.getenv("LLM_MODEL", "z-ai/glm-5.1")

# 字符串 → int 手动转,非数字会立刻 ValueError(也是启动时挂)
PORT = int(os.getenv("PORT", "12234"))


# ─── 复杂类型:os.getenv 只给字符串,list[str] 自己 split ───
_cors_raw = os.getenv("CORS_ALLOW_ORIGINS", "*").strip()
CORS_ORIGINS: list[str] = (
    ["*"] if _cors_raw == "*"
    else [x.strip() for x in _cors_raw.split(",") if x.strip()]
)


In [ ]:
# .env
OPENROUTER_API_KEY=sk-or-xxxxxxxxxxxxxxxxxxxxxxxx
LLM_MODEL=z-ai/glm-5.1
PORT=12234
API_KEY=my-real-secret-key
CORS_ALLOW_ORIGINS=http://localhost:12233,https://my-frontend.com

In [ ]:
# main.py 或 routers/ai.py
from fastapi import FastAPI
from .config import LLM_MODEL  # 模块顶层 import = 全进程天然单例

app = FastAPI()


# 前端请求示例:GET /info
# 后端返回示例:HTTP/1.1 200 OK + {"model": "z-ai/glm-5.1"}
@app.get("/info")
async def info() -> dict:
    # 配置在 import config.py 那一刻就已经加载完了,这里直接拿模块级常量用
    return {"model": LLM_MODEL}


- **必填字段用 `os.environ[KEY]` 启动校验**:缺字段直接 `KeyError`,部署脚本 / CI 立刻拦截。**这是"启动时挂 vs 运行时挂"的关键**——别用 `os.getenv("KEY")` 接必填字段,默认返 `None`,要等业务调到才发现服务起错,生产已经对外 500 十几分钟了。
- **类型转换手写**:`int(os.getenv("PORT", "8000"))` / `float(...)` / `os.getenv("DEBUG", "false").lower() == "true"`——啰嗦但很直白,非数字字符串调 `int()` 会立刻 `ValueError`,也算启动时挂的一种。
- **复杂类型自己 parse**:list 用 `.split(",")`、JSON 配置用 `json.loads(os.getenv("FOO", "{}"))`——多两三行 parse 代码而已,够用。
- **环境变量优先 `.env`**:`load_dotenv()` 默认**不覆盖**已经存在的 `os.environ` 字段——Docker / k8s 部署时直接通过容器环境变量覆盖,本地用 `.env`,容器用环境变量,两边天然分离。

&emsp;&emsp;<font color=red>注意多 worker 状态问题</font>:目前我们的会话历史 / 待办状态存在进程内字典或 SQLite。本机单进程跑得稳,但 gunicorn 多 worker 下每个 worker 独立内存,进程内 dict 互相看不见。生产环境应使用外部存储 / 数据库代替进程内 dict。

&emsp;&emsp;**这套"`config.py` 集中声明 + `python-dotenv` 加载 `.env`"是 FastAPI 项目里最朴素够用的配置层**——所有配置在一个模块里能扫到、`.env` 一键加载、必填字段启动校验。代码量低、依赖少,任何 router 直接 `from .config import XXX` 就用,改一处全局生效。

&emsp;&emsp;配置管完了,最后还有一类拓展层要补——失败响应。前面我们已经用了好几次 `HTTPException(404)` / `HTTPException(401)`,这一节简提一下完整用法。

### 14.4 失败响应的实现

&emsp;&emsp;前置课讲过 HTTP 状态码(`200 / 400 / 401 / 404 / 422 / 500` 各代表什么),但没讲怎么在 endpoint 里抛。FastAPI 提供 `HTTPException` 类,一行就完事——前面几个接口和鉴权小节已经用过好几次,这一节把它的完整用法简提一下。

&emsp;&emsp;**FastAPI 关键代码片段**:

In [ ]:
from fastapi import FastAPI, HTTPException, status

app = FastAPI()

_sessions = {"s1": [...]}  # mock


# 前端请求示例:GET /sessions/s_999/messages   ← s_999 不存在
# 后端返回示例:HTTP/1.1 404 Not Found
#         {"detail": "session s_999 not found"}
@app.get("/sessions/{session_id}/messages")
async def get_messages(session_id: str) -> dict:
    if session_id not in _sessions:
        # 最常见用法:状态码 + detail 字符串
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail=f"session {session_id} not found",
        )

    if len(_sessions[session_id]) > 1000:
        # 完整用法:状态码 + detail + 自定义响应头
        raise HTTPException(
            status_code=status.HTTP_413_REQUEST_ENTITY_TOO_LARGE,
            detail="session too large, please paginate",
            headers={"X-Limit": "1000"},
        )

    return {"messages": _sessions[session_id]}

&emsp;&emsp;**FastAPI 收到 `HTTPException` 后**:① 中断后续业务逻辑;② 按 `status_code` 设响应状态;③ 把 `detail` 包成 `{"detail": "..."}` 返回 JSON;④ 加上 `headers`(如果有)。客户端看到的就是标准 HTTP 错误响应。

> <font size=2>**【名词解释】<font color=red>HTTPException</font>(FastAPI 异常类)** — FastAPI 用来在路由内主动抛 4xx / 5xx 错误的异常类。构造参数 `status_code`(必填,HTTP 状态码)+ `detail`(任意可 JSON 序列化对象,通常是字符串 / dict)+ `headers`(响应头 dict,可选)。FastAPI 拦截这个异常生成对应的 `JSONResponse`,业务代码不用自己造响应对象。</font>

&emsp;&emsp;**HTTP 状态码语义速查**(本节代码用到 2-3 个,其余作参考):

<p align="center"><font face="黑体" size=4>HTTP 状态码语义速查</font></p>
<div align="center">
<table align="center" style="max-width: 820px; width: 100%;">
<thead>
<tr><th align="center">接口需求</th><th align="center">FastAPI 知识</th></tr>
</thead>
<tbody>
<tr><td align="left">HTTP 状态码语义</td><td align="left"><code>200 / 400 / 401 / 404 / 413 / 422 / 500</code></td></tr>
<tr><td align="left">抛 4xx / 5xx</td><td align="left"><code>raise HTTPException(status_code=..., detail=...)</code></td></tr>
<tr><td align="left">错误详情</td><td align="left"><code>detail=</code> 任意可序列化对象</td></tr>
<tr><td align="left">自定义响应头</td><td align="left"><code>headers={"X-...": "..."}</code></td></tr>
<tr><td align="left">状态码常量</td><td align="left"><code>status.HTTP_404_NOT_FOUND</code></td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**`HTTPException` 跟 try-except 的关系**:它们配合用——业务代码里业务异常用 try-except 包,捕获后转成 `HTTPException`:

In [ ]:
# 前端请求示例:POST /parse   Body: {"file_id": "missing-file"}
# 后端返回示例:HTTP/1.1 404 Not Found       ← FileNotFoundError → 404
#         {"detail": "file not found: missing-file"}
@app.post("/parse")
async def create_parse(req: ParseRequest):
    try:
        result = await do_something()
    except FileNotFoundError as exc:
        raise HTTPException(status_code=404, detail=f"file not found: {exc}")
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=f"bad input: {exc}")
    return result

&emsp;&emsp;<b>为什么本课不用 `exception_handler`?</b>FastAPI 还有 `@app.exception_handler(MyException)` 全局异常处理器,可以集中处理某类异常。本课规模够小,<b>每个 endpoint 自己 `raise HTTPException` 就足够了</b>,集中处理器留给更大型项目(几十个 endpoint + 同一类异常反复出现的场景)。

## 15. 本章速查表

&emsp;&emsp;本章已经把 11 类业务接口 + 4 类辅助组件的 FastAPI 关键写法都列出来了。这一节用一张表汇总:每一类接口的 HTTP 形态、对应的 FastAPI 组件、典型 endpoint 路径骨架。哪些是复用、哪些是新增,正文里已经解释过,表格里不再单列。

<p align="center"><font face="黑体" size=4>第 2 章接口实现速查表</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">接口 / 组件</th><th align="center">HTTP 形态</th><th align="center">FastAPI 组件</th></tr>
</thead>
<tbody>
<tr><td align="left">1</td><td align="left">会话历史 / 任务状态</td><td align="left"><code>GET + Path/Query</code></td><td align="left"><code>Path(...)</code> + <code>Query(...)</code> + <code>HTTPException(404)</code></td></tr>
<tr><td align="left">2</td><td align="left">CRUD</td><td align="left"><code>GET/POST/PATCH/PUT/DELETE</code> + 资源路由</td><td align="left"><code>APIRouter(prefix=..., tags=...)</code> + <code>app.include_router</code></td></tr>
<tr><td align="left">3</td><td align="left">聊天接口</td><td align="left"><code>POST + Body</code></td><td align="left"><code>@app.post</code> + <code>BaseModel</code> 嵌套(<code>list[Message]</code>)+ 自动 422</td></tr>
<tr><td align="left">4</td><td align="left">聊天流式</td><td align="left"><code>POST + StreamingResponse(text/plain)</code></td><td align="left"><code>StreamingResponse</code> + async generator + <code>agent.astream</code> chunks</td></tr>
<tr><td align="left">5</td><td align="left">工具结果 / Response 矩阵</td><td align="left"><code>GET + JSONResponse</code></td><td align="left">6 类 Response 子类对比表</td></tr>
<tr><td align="left">6</td><td align="left">SSE 多事件</td><td align="left"><code>GET + StreamingResponse(text/event-stream)</code></td><td align="left"><code>sse_pack</code> 工具 + 业务事件协议 + <code>astream_events(version="v2")</code></td></tr>
<tr><td align="left">7</td><td align="left">文件上传</td><td align="left"><code>POST + UploadFile + multipart/form-data</code></td><td align="left"><code>file: UploadFile = File(...)</code> + 分块读取</td></tr>
<tr><td align="left">8</td><td align="left">文件解析</td><td align="left"><code>POST + BackgroundTasks</code> + 三段式</td><td align="left"><code>BackgroundTasks</code> + <code>bg.add_task</code> + 状态存储 + <code>GET status</code> + <code>GET result</code></td></tr>
<tr><td align="left">9</td><td align="left">报告 / 导出下载</td><td align="left"><code>GET + FileResponse</code></td><td align="left"><code>FileResponse(path, filename=..., media_type=...)</code></td></tr>
<tr><td align="left">10</td><td align="left">长任务三段式(通用)</td><td align="left"><code>POST + GET + GET + DELETE</code></td><td align="left">复用 BackgroundTasks / 状态存储 + 取消接口</td></tr>
<tr><td align="left">11</td><td align="left">实时语音双向</td><td align="left"><code>WebSocket</code></td><td align="left"><code>@app.websocket</code> + <code>receive_bytes()</code> + <code>send_bytes()</code> + <code>WebSocketDisconnect</code></td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;我们扫一遍这张表,有几条规律可以记住:

- **核心 4 组件**:`@app.post/get/...` 路由 + `BaseModel` 请求体 + `Response` 子类(6 类)+ `Depends` 依赖,90% 的业务接口用这四样就够。
- **流式相关 3 组件**:`StreamingResponse(text/plain)`(普通流)+ `StreamingResponse(text/event-stream)`(SSE)+ `WebSocket`(双向)——按"单向纯文本 / 单向多事件 / 双向二进制"三档选。
- **异步任务 1 组件**:`BackgroundTasks` 一行注册,文件解析 / 长任务都用同一套三段式骨架。
- **辅助组件 4 件套**:`Depends` 鉴权 / `CORSMiddleware` 跨域 / `os.getenv` + `.env` 配置 / `HTTPException` 错误响应——任何一条上线接口都要带这四样。

# <center>第 3 章: ai-todo 案例 — 方法论在真实项目上的一次落地</center>

&emsp;&emsp;前两章我们一步一步把方法论搭好——第 1 章把 agent 类型归纳成 9 类、每一类列出业务接口出口;第 2 章用 6 维度框架把业务接口翻译成 HTTP 形态、再用 FastAPI 写成能跑的代码。**到这里手里已经有完整的"产品语言 → HTTP 形态 → FastAPI 组件"三步框架**——任何一套方法论,只有在真实项目上跑一遍才算落地。

&emsp;&emsp;这一章我们拿一个真实可跑的 agent 项目 — **ai-todo**(对话式待办管理 agent)— 走一遍完整方法论。本章不做代码逐行 walkthrough,**只做一件事:把前 2 章每一层方法论在 ai-todo 上印证一遍**。结构上分五节:**类型判断 → 9 条业务接口清单 → HTTP 形态映射 → 用了哪些 FastAPI 组件 → 哪些没用 / 为什么**,一一对应前 2 章。每一节的产出是一张表,把"产品形态 → 方法论 → 落地结果"摆给你看。

## 1. ai-todo 类型判断:对话 + 任务执行的混合体

&emsp;&emsp;ai-todo 是一个三栏布局网页应用:左边会话列表(像 ChatGPT)、中间对话主区、右边月历 + 当日待办;用户用自然语言"加一个明天交报告的待办" / "把 1 号标记完成" / "列出所有 todos",agent 解析意图、调工具、改数据、推回前端。可以直接按照《ai-todo安装与运行.md>文件进行安装运行。

&emsp;&emsp;**用第 1 章 9 类 agent 类型表对照一眼能看出来** —— ai-todo 不是单一类型,是**对话助手 + 工具调用的混合体**:对话能力(聊天 / 流式 / 会话历史 / 会话管理)对应对话助手那一节的接口出口;工具调用能力(`add_todo` / `update_todo` / `delete_todo` / `set_session_title` 等)对应工具调用那一节的"工具结果回流"出口。**类型判断决定了"接口骨架必须有什么、可以暂时没有什么"**——既然 ai-todo 不是数据分析 / RAG agent,它就不需要文件上传 / 长任务 / BackgroundTasks 这一组接口。这就是第 1 章"主形态 + 辅助形态"叠加规则在 ai-todo 上的第一次落地。

## 2. 9 条业务接口清单 — 第 1 章方法论的"答卷"

&emsp;&emsp;按第 1 章方法论,从两类主形态(对话助手 + 工具调用)的接口出口清单推出 ai-todo 的业务接口——共 **9 条**,按业务领域分三组:**对话(2 条)** + **会话管理(3 条)** + **待办对象(4 条)**:

<p align="center"><font face="黑体" size=4>ai-todo 的 9 条业务接口</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">业务接口名</th><th align="center">一句话说明</th><th align="center">类型出处</th></tr>
</thead>
<tbody>
<tr><td align="left">①</td><td align="left"><strong>聊天流式接口</strong></td><td align="left">用户敲一句话,agent 一边推理一边把回复一字一字流回前端(纯文本流)</td><td align="left">对话助手</td></tr>
<tr><td align="left">②</td><td align="left"><strong>工具调用过程接口</strong></td><td align="left">前端看到"AI 正在调 <code>add_todo</code> / 返了 id=42"的结构化事件流</td><td align="left">工具调用</td></tr>
<tr><td align="left">③</td><td align="left"><strong>会话历史接口</strong></td><td align="left">切换到一条老会话时拉之前的所有消息</td><td align="left">对话助手</td></tr>
<tr><td align="left">④</td><td align="left"><strong>会话删除接口</strong></td><td align="left">删一条会话及其下所有消息</td><td align="left">对话助手</td></tr>
<tr><td align="left">⑤</td><td align="left"><strong>会话标题接口</strong></td><td align="left">agent 自动生成的会话标题,前端定期拉来更新左栏</td><td align="left">对话助手</td></tr>
<tr><td align="left">⑥</td><td align="left"><strong>待办列表接口</strong></td><td align="left">前端拉某段时间的待办列表渲染到右栏</td><td align="left">业务对象 CRUD</td></tr>
<tr><td align="left">⑦</td><td align="left"><strong>待办更新接口</strong></td><td align="left">改某一条待办的部分字段(切换完成 / 自然语言改)</td><td align="left">业务对象 CRUD</td></tr>
<tr><td align="left">⑧</td><td align="left"><strong>待办删除接口</strong></td><td align="left">删一条待办</td><td align="left">业务对象 CRUD</td></tr>
<tr><td align="left">⑨</td><td align="left"><strong>待办导出接口</strong></td><td align="left">导出全部待办为 JSON 文件下载到本地</td><td align="left">拓展(产物导出)</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**业务接口清单只用产品语言**,这一步还没到协议层——协议层应该从业务需求推出来。一个关键观察:**这 9 条接口完全用对话助手 + 工具调用 + 业务对象 CRUD 三组方法论就覆盖了**,没有触发文件上传 / 长任务 / WebSocket。这是第 1 章方法论的一次完整答卷——类型判断对了,接口清单自然收敛。

## 3. 业务接口 → HTTP 形态 — 第 2 章方法论的 9 行答卷

&emsp;&emsp;把上一节的 9 条业务接口逐条过第 2 章的 **6 维度框架**(输入复杂度 / 是否改状态 / 是否需要实时 / 是否需要文件 / 是否长任务 / 协议升级),9 行答卷直接落到 HTTP 形态:

<p align="center"><font face="黑体" size=4>ai-todo 9 条业务接口 → HTTP 形态完整翻译</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">业务接口</th><th align="center">6 维度命中</th><th align="center">HTTP 形态</th><th align="center">回扣第 2 章</th></tr>
</thead>
<tbody>
<tr><td align="left">①</td><td align="left">聊天流式接口</td><td align="left">输入复杂(messages + session_id)/ 创建 / <strong>实时</strong>(token 级)/ 纯文本</td><td align="left"><code>POST /chat/stream</code> + <code>Body</code> + <code>StreamingResponse(text/plain)</code></td><td align="left">流式回复接口</td></tr>
<tr><td align="left">②</td><td align="left">工具调用过程接口</td><td align="left">输入复杂 / 创建 / <strong>实时</strong> / <strong>结构化事件</strong>(tool_call / tool_result / message_delta 等 7 类)</td><td align="left"><code>GET /chat/events</code> + Query + <code>StreamingResponse(text/event-stream)</code>(SSE)</td><td align="left">工具调用过程接口(SSE)</td></tr>
<tr><td align="left">③</td><td align="left">会话历史接口</td><td align="left">简单(一个 session_id)/ 只读</td><td align="left"><code>GET /sessions/{session_id}/messages</code> + Path 参数</td><td align="left">会话历史接口</td></tr>
<tr><td align="left">④</td><td align="left">会话删除接口</td><td align="left">简单(一个 session_id)/ 删除</td><td align="left"><code>DELETE /sessions/{session_id}</code></td><td align="left">CRUD(D 部分)</td></tr>
<tr><td align="left">⑤</td><td align="left">会话标题接口</td><td align="left">简单(一个 session_id)/ 只读</td><td align="left"><code>GET /sessions/{session_id}/title</code> + Path 参数</td><td align="left">会话状态查询</td></tr>
<tr><td align="left">⑥</td><td align="left">待办列表接口</td><td align="left">简单(可选 <code>only_pending</code> 过滤)/ 只读</td><td align="left"><code>GET /todos</code> + Query</td><td align="left">CRUD(R 部分,列表)</td></tr>
<tr><td align="left">⑦</td><td align="left">待办更新接口</td><td align="left">中等(Path 一个 id + Body 部分字段)/ 部分改</td><td align="left"><code>PATCH /todos/{todo_id}</code> + Path + Body</td><td align="left">CRUD(U 部分,部分改用 PATCH 不用 PUT)</td></tr>
<tr><td align="left">⑧</td><td align="left">待办删除接口</td><td align="left">简单(一个 id)/ 删除</td><td align="left"><code>DELETE /todos/{todo_id}</code></td><td align="left">CRUD(D 部分)</td></tr>
<tr><td align="left">⑨</td><td align="left">待办导出接口</td><td align="left">简单 / 只读 / <strong>涉及文件(下载)</strong></td><td align="left"><code>GET /todos/export</code> + <code>FileResponse</code></td><td align="left">报告 / 导出下载接口</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;横扫这张表两条规律值得记住——① **半数接口是普通 RESTful CRUD**(③④⑤⑥⑦⑧):`GET/PATCH/DELETE` + Path/Query 就够,这是第 2 章 CRUD 那节最朴素的形态;② **流式接口分两条**(① 纯文本流 vs ② SSE 多事件)对应"客户端要不要按事件类型分发"那一维度——**两条都需要,不合并**,因为纯字流场景前端不该多绕一层"解 SSE 报文"。每一行的"6 维度命中"列就是方法论在 ai-todo 上的直接印证。

## 4. 用了哪些 FastAPI 组件 — 第 2 章 FastAPI 实现部分的落地清单

&emsp;&emsp;对照第 2 章末尾速查表,ai-todo 真实用到 **8 件 FastAPI 组件**——5 件实现业务接口 + 3 件辅助组件 + `HTTPException` 处理错误响应:

<p align="center"><font face="黑体" size=4>完整组件清单表</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">#</th><th align="center">FastAPI 组件</th><th align="center">ai-todo 在哪用</th><th align="center">落到哪 9 条业务接口</th></tr>
</thead>
<tbody>
<tr><td colspan="4" align="left"><strong>━━━ 实现业务接口的 5 件 ━━━</strong></td></tr>
<tr><td align="left">1</td><td align="left"><strong>HTTP Method 选型</strong>(GET / POST / PATCH / DELETE)</td><td align="left"><code>routers/chat.py</code> / <code>sessions.py</code> / <code>todos.py</code> 各 router 上</td><td align="left">① POST / ②⑤⑥⑨ GET / ⑦ PATCH / ④⑧ DELETE</td></tr>
<tr><td align="left">2</td><td align="left"><strong>Pydantic 嵌套 schema</strong>(<code>ChatRequest</code> / <code>Todo</code> / <code>TodoPatch</code> / <code>Message</code>)</td><td align="left"><code>models.py</code> 4 个 BaseModel</td><td align="left">① 入参 / ③⑥⑦ 出参</td></tr>
<tr><td align="left">3</td><td align="left"><strong>Response 类型</strong>(JSONResponse 默认 / StreamingResponse / FileResponse 三种都用)</td><td align="left">三种全用上 — JSON 默认 / Streaming 用于 ①② / FileResponse 用于 ⑨</td><td align="left">全部 9 条</td></tr>
<tr><td align="left">4</td><td align="left"><strong><code>StreamingResponse(text/plain)</code></strong>(普通流式)</td><td align="left"><code>routers/chat.py</code> <code>chat_stream</code> 函数</td><td align="left">① 聊天流式接口</td></tr>
<tr><td align="left">5</td><td align="left"><strong>SSE(<code>text/event-stream</code>)+ 业务事件协议</strong></td><td align="left"><code>routers/chat.py</code> <code>chat_events</code> + <code>sse.py</code> <code>sse_pack</code> + <code>translate</code></td><td align="left">② 工具调用过程接口</td></tr>
<tr><td colspan="4" align="left"><strong>━━━ 完整接口必备的 4 件辅助 ━━━</strong></td></tr>
<tr><td align="left">6</td><td align="left"><strong><code>Depends</code> + 鉴权</strong>(<code>verify_api_key</code>)</td><td align="left"><code>auth.py</code> + <code>main.py</code> <code>app.include_router(..., dependencies=auth_dep)</code></td><td align="left">全部 9 条都带</td></tr>
<tr><td align="left">7</td><td align="left"><strong><code>CORSMiddleware</code></strong></td><td align="left"><code>main.py</code> <code>app.add_middleware(CORSMiddleware, ...)</code></td><td align="left">全部 9 条都覆盖</td></tr>
<tr><td align="left">8</td><td align="left"><strong><code>os.getenv</code> + <code>python-dotenv</code></strong></td><td align="left"><code>config.py</code> 集中声明 + <code>.env</code></td><td align="left">全局配置(端口 / API key / 模型 / .env)</td></tr>
<tr><td align="left">9</td><td align="left"><strong><code>HTTPException</code></strong>(错误响应)</td><td align="left">各 router 零散用(<code>404</code> / <code>401</code>)</td><td align="left">各业务接口的失败分支</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;一个核心结论:**章 2 讲了 16 类组件,ai-todo 用了 8 件,集中在"对话 + 工具调用 + CRUD"三组**——这印证了**类型判断决定接口骨架**:一个对话助手 + 工具调用混合体的 agent,需要的组件就是这一组,不会多也不会少。第 2 章不是教你"FastAPI 所有东西都要用上",而是给你一张完整零件清单 + 一套挑零件的方法论。

## 5. 哪些没用 / 为什么 — 方法论的反向验证

&emsp;&emsp;前 4 节一直在讲"ai-todo 用了什么",这一节反过来——**第 2 章讲了 11 类业务接口 + 4 件辅助组件,ai-todo 只用了 6 类业务接口 + 4 件辅助**,WebSocket / UploadFile + 文件解析 / BackgroundTasks 长任务三段式整组都没用。每一件"没用"都能用 6 维度反推清楚:

<p align="center"><font face="黑体" size=4>ai-todo 没用的三件组件 — 6 维度反推</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">没用的组件</th><th align="center">6 维度命中(决定性维度)</th><th align="center">什么时候要用上</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>WebSocket</strong></td><td align="left">维度 1 双向 — <strong>否</strong>(客户端不需要在不发新请求的前提下主动推数据 → SSE 单向够用)</td><td align="left">实时语音 agent / 多人协作 / 实时游戏(双向 + 二进制 + 高频)</td></tr>
<tr><td align="left"><strong>UploadFile + multipart</strong></td><td align="left">维度 5 文件 — <strong>否(上传方向)</strong>(所有输入都是对话文本,没有用户上传文件场景)</td><td align="left">数据分析 agent(传 CSV)/ 文件处理 agent(传 PDF)/ RAG 上传文档</td></tr>
<tr><td align="left"><strong>BackgroundTasks 长任务三段式</strong></td><td align="left">维度 6 长任务 — <strong>否</strong>(单次对话 + 调一两个工具 ≤ 30 秒,流式响应就能撑住)</td><td align="left">代码 agent 跑改造任务 / 数据分析跑大表 / 工作流 agent 多节点流程</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**所有"没用"全部走 6 维度反推 → 跟产品形态对齐** — 这就是方法论"反向验证"的价值:**你不会因为"FastAPI 教程里讲了 WebSocket / BackgroundTasks"就觉得自己的 agent 应该用上**。能不能说清"为什么不用",比能说清"为什么用"更能体现一个工程师对产品的判断力。

&emsp;&emsp;走到这里,前 3 章方法论已经在 ai-todo 真实项目上完整跑通一遍。**但这套方法论不是 ai-todo 专属**——换一个 agent 类型,同样的五步走法,得到的接口设计图不同。我们用一个 **企业知识库问答 agent**(RAG 类,类似 Perplexity / Glean 的内部版)快速演示一遍:

<p align="center"><font face="黑体" size=4>这套方法论可以用在任何agent设计上</font></p>
<div align="center">
<table align="center">
<thead>
<tr><th align="center">方法论步骤</th><th align="center">这个 RAG 项目上怎么走</th></tr>
</thead>
<tbody>
<tr><td align="left"><strong>第一步:类型判断</strong></td><td align="left">落在 RAG / 知识库 agent 主形态;同时叠加对话助手能力包(聊天 + 流式 + 会话管理)</td></tr>
<tr><td align="left"><strong>第二步:业务接口清单</strong></td><td align="left">用 RAG 主形态出口 + 对话基础包,推出 11 条业务接口:聊天 + 流式 + <strong>回复带引用</strong> + 会话 3 件套(列 / 删 / 改名)+ <strong>文档上传</strong> + <strong>文档解析进度</strong> + <strong>文档管理(增删改查)</strong> + <strong>检索结果预览</strong>(高级)</td></tr>
<tr><td align="left"><strong>第三步:HTTP 形态翻译</strong></td><td align="left">拿 11 条业务接口跑第 2 章 6 维度:聊天 → POST + Body / 流式 → StreamingResponse / 回复带引用 → SSE 多事件(引用作 <code>citation</code> 事件)/ 文档上传 → UploadFile / 文档解析 → 长任务三段式 + BackgroundTasks / 检索结果预览 → GET + Query</td></tr>
<tr><td align="left"><strong>第四步:FastAPI 组件</strong></td><td align="left">比 ai-todo 多出 4 件——<code>UploadFile</code>/ <code>BackgroundTasks</code>/ <code>FileResponse</code>(文档下载场景)/ 向量数据库客户端(接口层无关)</td></tr>
<tr><td align="left"><strong>第五步:反向验证</strong></td><td align="left">没用 <code>WebSocket</code>(单向推送够);没用 <code>task_id 三段式</code> 的“取消”(用户通常不取消文档上传);<strong>用了进程内 dict?不,直接上外部数据库</strong>(知识库场景多用户多会话,从一开始就要外置状态)</td></tr>
</tbody>
</table>
</div>

&emsp;&emsp;**这就是这套方法论真正可迁移的样子**——同样的五步骨架,套到不同 agent 类型上,得到的接口设计图不同。你手上的任何 agent 项目(代码 agent / 数据分析 agent / 工作流 agent / RAG / 多智能体 ...)都能按这个流程走一遍,得到一张完整的"产品形态 → 业务接口 → HTTP 形态 → FastAPI 组件"对照图。这是本课交付给你的最重要的工程能力。

# <center>第 4 章: 回顾总结 — Agent 接口设计 5 步方法论</center>

&emsp;&emsp;**本章负责把整套方法论沉淀成一张可以拍下来贴墙上的 5 步公式 + 三层抽象总图**。以后拿到任何一个 agent 项目,都能照这套走一遍,推出接口设计骨架。

<div align=center><font size=2 color=#999999>本章 3 件事:5 步公式 → 三层抽象总图 → 接口装好了下一步去哪</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-00362ae4.jpg" width=80%></div>
<br>

## 1. 5 步公式:从 agent 项目到接口设计骨架

<div align=center><font size=2 color=#999999>Agent 接口设计 5 步公式 — 5 步顺序流程 + 每步产出物 + 每步对应章节查表位置</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-5fce8c8f.jpg" width=85%></div>
<br>

&emsp;&emsp;<font color=red>这五步是顺序的,不能跳</font>——第一步不做就开始拆出口、第二步不做就开始选协议、第三步不做就开始写代码,每一步都会埋下不可逆的设计错误。**举个最常见的反例**:聊天接口跳过第 3 步直接用 `GET + Query` 写——一周后产品改成"传整段对话历史",Query 撑不下、`?messages=[...]` URL 超长被反代截断,只能把所有路由从 `GET` 全部翻成 `POST + Body`,前端跟着翻、API doc 跟着翻、客户端 SDK 跟着翻。**前三步是"想"的成本,第四步是"写"的成本——想的成本投入越多,写的成本越低,后期改的成本最低**。

&emsp;&emsp;任何 agent 项目过来,五步走一遍能直接落到接口设计骨架。**第 3 章的 ai-todo 案例就是这五步公式跑出来的完整答卷**——五步对应得清清楚楚。同样的五步换成知识库 / 代码 / 数据分析 agent,接口设计图完全不同,**但骨架走法一致**。

## 2. 三层抽象总图回顾

<div align=center><font size=2 color=#999999>Agent 接口设计三层抽象总图 — 业务出口层 → HTTP 协议层 → FastAPI 实现层(第 3 章 ai-todo 案例串通三层验证)</font></div>
<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/Leo/agent-api-design/2026-05-15/img-1a75b82c.jpg" width=85%></div>
<br>

&emsp;&emsp;**5 步公式 vs 三层抽象的关系**:5 步公式是"怎么走"——从一个项目走到接口骨架的顺序路径;三层抽象是"长什么样"——每条接口在上中下三层各占什么位置。**两张图配着看**:写接口的时候按 5 步走,review 接口的时候按三层抽象检查"上中下各层有没有漏"。